<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/11.%20Agentic%20RAG%20%26%20GraphRAG/L10_agentic_rag_and_graphrag.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Agentic RAG and GraphRAG

This chapter treats retrieval as an engineering discipline rather than a single library call. It is written for engineers who already build production systems with Python, asynchronous code, Docker, LangChain and LangGraph, multi-agent frameworks, embeddings, and conventional retrieval-augmented generation. It assumes fluency with vector databases, Neo4j, and SQL, and it does not restate those foundations. The subject here is what happens when a naive top-k retriever meets a real enterprise knowledge base and quietly returns wrong answers, and what an engineer must build instead.

The material is organized as a sequence of parts. Each part opens with its learning objectives and a concrete motivation, develops the architecture and the underlying concept in prose, renders the relevant structure as an ASCII diagram, and then grounds the discussion in runnable code. Where a part carries recurring engineering hazards, a paragraph on common mistakes and one on production practice follow, together with interview-style questions and their model answers. A single worked question threads through the first several parts so that the reader can watch one hard query defeat naive retrieval and then be answered correctly by an agentic, graph-aware system.

The notebook is designed to run in Google Colab and to remain useful as standalone reference material afterward. Two design decisions make it reproducible. First, the relational demonstrations use SQLite from the standard library and the conceptual graph demonstrations use an in-memory property graph built on NetworkX, so those cells execute with no external accounts. Second, the parts that require a graph database are written against Neo4j with real Cypher and the official driver, and they execute against Neo4j Aura Free or a local Community Edition server when connection details are supplied; when they are not, those cells print the Cypher they would run and fall back to the in-memory engine so that the notebook still produces answers end to end. The agentic components call the Groq API when a key is present and otherwise use a deterministic offline client that returns representative outputs for the worked examples. None of these fallbacks change the shape of the code an engineer would ship; they exist so the chapter can be read and run without procurement.

## Chapter setup

The setup section installs dependencies, configures optional credentials, and constructs the objects reused throughout the chapter: a chat wrapper over an instruction-tuned model, an embedding utility, a Neo4j connection helper, a small fictional organization used as the running dataset, and an in-memory property graph that mirrors that organization. Running the setup cells in order is a prerequisite for every later part.

### Setting up Neo4j (optional)

The graph-database cells run against a real Neo4j instance when one is configured, and otherwise fall back to an in-memory graph so the notebook still runs end to end. To use a real instance, create a **free Neo4j Aura** database:

1. Go to [console.neo4j.io](https://console.neo4j.io) and sign in.
2. Click **Create instance** and choose **AuraDB Free**.
3. When it provisions, a credentials pane appears **once** — download the `Neo4j-xxxx-Created-....txt` file (or copy the password). The password is shown only this one time.
4. Wait until the instance status is **RUNNING** (this can take a few minutes).

That file contains everything you need:

```
NEO4J_URI=neo4j+s://xxxxxxxx.databases.neo4j.io
NEO4J_USERNAME=xxxxxxxx      # often the instance id, not the literal "neo4j"
NEO4J_PASSWORD=................
```

**Provide the credentials in any one of these ways** (the next cell reads the environment first, then prompts for anything missing):

- **Prompts (recommended):** just run the next cell and paste the values when asked — the password is entered with `getpass`, so it is never written into the notebook.
- **Environment / `neo4j.env`:** `export NEO4J_URI=... NEO4J_USERNAME=... NEO4J_PASSWORD=...` before launching Jupyter, then run the next cell (it will not prompt for values already set).
- **Colab:** store them in the **Secrets** panel and load them into `os.environ`.

> Use the **exact** username from the credentials file. On many Aura Free instances it is the instance id (e.g. `df1b8ee7`), not `neo4j`.

In [3]:
# Dependencies. Safe to re-run. SentenceTransformers pulls PyTorch, which is
# already present in Colab; the first embedding call downloads a small model.
!pip -q install "neo4j>=5,<6" sentence-transformers groq networkx pandas numpy
print("Dependencies installed.")

Dependencies installed.


In [4]:
# Standard imports and secure credential entry.
#
# Nothing sensitive is stored in this notebook. Each credential is read from the
# environment first (e.g. exported from a neo4j.env file or set via the Colab
# Secrets panel); if it is missing, you are prompted for it at runtime. Secrets
# use getpass, so the value is never echoed or saved into the .ipynb.
#
# Every credential is optional: press Enter to skip any prompt. Without a Groq
# key the agentic cells use a deterministic fallback, and without Neo4j details
# the graph-database cells fall back to the in-memory engine.
import os
import json
from getpass import getpass


def _prompt(var, prompt, secret=False):
    """Fill os.environ[var] from a prompt unless it is already set. Blank skips."""
    if os.environ.get(var):
        return
    value = (getpass(prompt) if secret else input(prompt)).strip()
    if value:
        os.environ[var] = value


# Groq: enables live LLM calls; skip to use the offline deterministic fallback.
_prompt("GROQ_API_KEY", "Groq API key (blank to skip): ", secret=True)

# Neo4j Aura: skip any prompt to fall back to the in-memory graph.
_prompt("NEO4J_URI", "Neo4j URI e.g. neo4j+s://xxxx.databases.neo4j.io (blank to skip): ")
_prompt("NEO4J_USERNAME", "Neo4j username (see credentials file; blank = 'neo4j'): ")
_prompt("NEO4J_PASSWORD", "Neo4j password (blank to skip): ", secret=True)

# Default the username only when a URI was supplied without one.
if os.environ.get("NEO4J_URI") and not os.environ.get("NEO4J_USERNAME"):
    os.environ["NEO4J_USERNAME"] = "neo4j"

print("Groq key present:", bool(os.environ.get("GROQ_API_KEY")))
# print("Neo4j configured:", bool(os.environ.get("NEO4J_URI") and os.environ.get("NEO4J_PASSWORD")))

Neo4j URI e.g. neo4j+s://xxxx.databases.neo4j.io (blank to skip): 
Neo4j username (see credentials file; blank = 'neo4j'): 
Neo4j password (blank to skip): ··········
Groq key present: True


In [5]:
# Chat wrapper over Groq with a deterministic offline fallback.
#
# The wrapper exposes a single complete(system, user, task) method. When a
# Groq key is available it calls the hosted model; otherwise it returns a
# canned, representative response selected by the task hint so that the
# planning, SQL-generation, extraction, and synthesis demonstrations still run.

_FALLBACK_PLAN = '''{"subquestions": [
  {"id": "q1", "tool": "vector", "text": "Identify projects whose documents describe LLM security work."},
  {"id": "q2", "tool": "graph",  "text": "Find employees assigned to those projects and the managers they report to."},
  {"id": "q3", "tool": "graph",  "text": "Keep only managers who belong to engineering."}
]}'''

_FALLBACK_SQL_CTO = '''{"sql": "WITH RECURSIVE org(id, name, manager_id, depth) AS (SELECT id, name, manager_id, 0 FROM employees WHERE title = 'CTO' UNION ALL SELECT e.id, e.name, e.manager_id, org.depth + 1 FROM employees e JOIN org ON e.manager_id = org.id) SELECT name FROM org WHERE depth > 0 ORDER BY name"}'''

_FALLBACK_SQL_BUDGET = '''{"sql": "SELECT p.name AS project, b.amount_usd FROM projects p JOIN budgets b ON b.project_id = p.id WHERE b.amount_usd > 500000 ORDER BY b.amount_usd DESC"}'''

_FALLBACK_SQL_UNDER_MANAGER = '''{"sql": "SELECT e.name FROM employees e JOIN employees m ON e.manager_id = m.id WHERE m.name = 'Elena Duarte' ORDER BY e.name"}'''

_FALLBACK_SQL_DEFAULT = '''{"sql": "SELECT name, title, department FROM employees ORDER BY name LIMIT 25"}'''

_FALLBACK_EXTRACT = '''{"entities": [
  {"name": "Guardrail", "type": "Project"},
  {"name": "prompt injection", "type": "Topic"},
  {"name": "LLM Security", "type": "Skill"}
], "relations": [
  {"source": "Guardrail", "type": "ADDRESSES", "target": "prompt injection"},
  {"source": "Guardrail", "type": "REQUIRES_SKILL", "target": "LLM Security"}
]}'''

_FALLBACK_ANSWER = ('''Based on the retrieved evidence, the engineering manager whose direct reports work on '''
                    '''LLM security projects is Elena Duarte. Her reports Ana Iyer and Fatima Zahra work on '''
                    '''Project Guardrail, and Ben Cho works on Project Sentinel, both of which are LLM security '''
                    '''efforts. Sofia Klein also manages people on these projects, but she leads Security rather '''
                    '''than Engineering and is therefore excluded by the engineering-manager constraint.''')


class LLM:
    '''Minimal chat interface with an offline fallback for reproducibility.'''

    def __init__(self, model="llama-3.3-70b-versatile"):
        self.model = model
        self.client = None
        key = os.environ.get("GROQ_API_KEY")
        if key:
            try:
                from groq import Groq
                self.client = Groq(api_key=key)
            except Exception as exc:  # pragma: no cover - environment dependent
                print("Groq client unavailable; using offline fallback:", exc)

    @property
    def live(self):
        return self.client is not None

    def complete(self, system, user, task=None, temperature=0.0):
        if self.client is not None:
            resp = self.client.chat.completions.create(
                model=self.model,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return resp.choices[0].message.content
        return _fallback_response(task, user)


def _fallback_response(task, user):
    text = (user or "").lower()
    if task == "plan":
        return _FALLBACK_PLAN
    if task == "sql":
        # Key only on the question text. The schema embedded in the prompt names
        # a 'budgets' table, which would otherwise match every SQL request.
        if "question:" in text:
            text = text.split("question:", 1)[1]
        if "cto" in text:
            return _FALLBACK_SQL_CTO
        if "budget" in text:
            return _FALLBACK_SQL_BUDGET
        if "report" in text and "manager" in text:
            return _FALLBACK_SQL_UNDER_MANAGER
        return _FALLBACK_SQL_DEFAULT
    if task == "extract":
        return _FALLBACK_EXTRACT
    if task == "synthesize":
        return _FALLBACK_ANSWER
    return "No offline fallback is defined for this task."


def parse_json(text):
    '''Extract the first JSON object from a model response, tolerating fences.'''
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if "\n" in cleaned:
            first, rest = cleaned.split("\n", 1)
            if first.strip().lower() in ("json", ""):
                cleaned = rest
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return json.loads(cleaned)


llm = LLM()
print("LLM mode:", "live (Groq)" if llm.live else "offline deterministic fallback")

LLM mode: live (Groq)


In [6]:
# Embedding utilities. A single small, normalized-output model is loaded lazily
# and cached. Cosine similarity reduces to a dot product on normalized vectors.
import numpy as np

_EMBED_MODEL = None


def _get_embed_model():
    global _EMBED_MODEL
    if _EMBED_MODEL is None:
        from sentence_transformers import SentenceTransformer
        _EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    return _EMBED_MODEL


def embed_texts(texts):
    # Return an (n, d) float32 array of L2-normalized embeddings.
    model = _get_embed_model()
    vectors = model.encode(list(texts), normalize_embeddings=True)
    return np.asarray(vectors, dtype="float32")


def cosine_topk(query_vec, matrix, k):
    # matrix and query_vec are normalized; scores are dot products.
    scores = matrix @ query_vec
    order = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in order]


print("Embedding utilities ready. The model downloads on first use.")

Embedding utilities ready. The model downloads on first use.


In [7]:
# Neo4j connection helper. Cells that target Neo4j guard on neo4j_available()
# and degrade to the in-memory engine when no instance is configured.

def neo4j_available():
    return bool(os.environ.get("NEO4J_URI") and os.environ.get("NEO4J_PASSWORD"))


_NEO4J_DRIVER = None


def get_driver():
    global _NEO4J_DRIVER
    if _NEO4J_DRIVER is None:
        from neo4j import GraphDatabase
        _NEO4J_DRIVER = GraphDatabase.driver(
            os.environ["NEO4J_URI"],
            auth=(os.environ.get("NEO4J_USERNAME", "neo4j"), os.environ["NEO4J_PASSWORD"]),
        )
    return _NEO4J_DRIVER


def run_cypher(query, **params):
    # Execute a Cypher statement and return records as a list of dicts.
    with get_driver().session() as session:
        return [record.data() for record in session.run(query, **params)]


if neo4j_available():
    try:
        print("Neo4j connectivity check:", run_cypher("RETURN 1 AS ok")[0])
    except Exception as exc:  # pragma: no cover - environment dependent
        print("Neo4j configured but unreachable:", exc)
else:
    print("Neo4j not configured. Graph-database cells will print their Cypher")
    print("and fall back to the in-memory graph so the notebook still runs.")

Neo4j not configured. Graph-database cells will print their Cypher
and fall back to the in-memory graph so the notebook still runs.


### The shared dataset

Every part operates on one small fictional company so that results can be verified by inspection. The organization has a reporting hierarchy rooted at a chief executive, several departments, a set of projects with budgets, per-employee skills, project assignments, and a handful of internal documents. The dataset is deliberately compact but structurally complete: it contains a multi-level management chain, cross-departmental project staffing, and unstructured text whose meaning is not captured by any single column. That combination is what makes the running question hard, and it is representative of the shape of real enterprise knowledge even though the scale is not.

In [8]:
# The shared organizational dataset. Tuples are used so the same rows can seed
# SQLite, the in-memory graph, and Neo4j without transformation.
import pandas as pd

DEPARTMENTS = ["Executive", "Engineering", "Security", "Data", "Design"]

# (id, name, title, department, manager_id, is_engineering)
EMPLOYEES = [
    (1,  "Nadia Osei",   "CEO",                 "Executive",   None, 0),
    (2,  "Priya Raman",  "CTO",                 "Engineering", 1,    1),
    (3,  "Marco Silva",  "VP Engineering",      "Engineering", 2,    1),
    (4,  "Elena Duarte", "Engineering Manager", "Engineering", 3,    1),
    (5,  "Sofia Klein",  "Security Manager",    "Security",    2,    0),
    (6,  "Tomas Berg",   "Data Manager",        "Data",        2,    0),
    (7,  "Ana Iyer",     "Senior Engineer",     "Engineering", 4,    1),
    (8,  "Ben Cho",      "Engineer",            "Engineering", 4,    1),
    (9,  "Chika Mba",    "Security Engineer",   "Security",    5,    0),
    (10, "Diego Rossi",  "Data Scientist",      "Data",        6,    0),
    (11, "Fatima Zahra", "ML Engineer",         "Engineering", 4,    1),
    (12, "Grace Lin",    "Designer",            "Design",      13,   0),
    (13, "Hiro Tanaka",  "Design Manager",      "Design",      2,    0),
    (15, "Omar Farouk",  "Engineering Manager", "Engineering", 3,    1),
    (16, "Iris Wong",    "Engineer",            "Engineering", 15,   1),
]

# employee_id -> list of skills
SKILLS = {
    4:  ["Leadership", "Python"],
    7:  ["Python", "LLM Security"],
    8:  ["Rust", "Cryptography"],
    9:  ["LLM Security", "Threat Modeling"],
    10: ["Python", "Embeddings"],
    11: ["LLM Security", "Python"],
    16: ["Go", "Kubernetes"],
}

# (id, name, department, annual_budget_usd)
PROJECTS = [
    (101, "Guardrail", "Security",    750000),
    (102, "Atlas",     "Data",        1200000),
    (103, "Prism",     "Design",      300000),
    (104, "Sentinel",  "Security",    600000),
    (105, "Nimbus",    "Engineering", 900000),
]

# (employee_id, project_id, role)
ASSIGNMENTS = [
    (7, 101, "Lead"), (9, 101, "Contributor"), (11, 101, "Contributor"),
    (10, 102, "Lead"), (7, 102, "Contributor"),
    (12, 103, "Lead"),
    (9, 104, "Lead"), (8, 104, "Contributor"),
    (16, 105, "Lead"), (15, 105, "Sponsor"),
]

# (id, project_id_or_None, title, text)
DOCUMENTS = [
    (201, 101, "Guardrail Overview",
     "Project Guardrail hardens the company LLM endpoints against prompt injection and "
     "jailbreak attacks. It covers input sanitization, output filtering, and periodic red "
     "teaming. This is core LLM security work owned by the Security department."),
    (202, 104, "Sentinel Overview",
     "Sentinel provides continuous LLM security monitoring of production model traffic, "
     "detecting adversarial prompts and potential data exfiltration in real time."),
    (203, 102, "Atlas Overview",
     "Atlas is the unified data platform providing feature storage, batch pipelines, and "
     "governed access to analytics tables across the company."),
    (204, 105, "Nimbus Overview",
     "Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh "
     "that host internal applications."),
    (205, 103, "Prism Overview",
     "Prism is the shared design system defining components, typography, and accessibility "
     "guidelines used across product surfaces."),
    (206, None, "Model Input Security Policy",
     "Company policy requires every project that handles model inputs to complete an LLM "
     "security threat model and to document mitigations for prompt injection before launch."),
]


def dataset_summary():
    frame = pd.DataFrame(
        [(e[0], e[1], e[2], e[3], e[4]) for e in EMPLOYEES],
        columns=["id", "name", "title", "department", "manager_id"],
    )
    print("Employees: {} | Departments: {} | Projects: {} | Documents: {}".format(
        len(EMPLOYEES), len(DEPARTMENTS), len(PROJECTS), len(DOCUMENTS)))
    return frame


dataset_summary()

Employees: 15 | Departments: 5 | Projects: 5 | Documents: 6


,id,name,title,department,manager_id
0,1,Nadia Osei,CEO,Executive,NaN
1,2,Priya Raman,CTO,Engineering,1.0
2,3,Marco Silva,VP Engineering,Engineering,2.0
3,4,Elena Duarte,Engineering Manager,Engineering,3.0
4,5,Sofia Klein,Security Manager,Security,2.0
5,6,Tomas Berg,Data Manager,Data,2.0
6,7,Ana Iyer,Senior Engineer,Engineering,4.0
7,8,Ben Cho,Engineer,Engineering,4.0
8,9,Chika Mba,Security Engineer,Security,5.0
9,10,Diego Rossi,Data Scientist,Data,6.0


In [14]:
PROJECTS

[(101, 'Guardrail', 'Security', 750000),
 (102, 'Atlas', 'Data', 1200000),
 (103, 'Prism', 'Design', 300000),
 (104, 'Sentinel', 'Security', 600000),
 (105, 'Nimbus', 'Engineering', 900000)]

In [10]:
# In-memory property graph mirroring the dataset. Node keys are (kind, id)
# tuples; edges carry a 'rel' attribute using the same relationship names as
# the Neo4j model introduced later. This engine backs the conceptual graph
# demonstrations and the fallback path when Neo4j is not configured.
import networkx as nx


class InMemoryOrgGraph:
    def __init__(self, employees, departments, skills, projects, assignments, documents):
        self.g = nx.MultiDiGraph()
        for dept in departments:
            self.g.add_node(("department", dept), kind="department", name=dept)
        for eid, name, title, dept, mgr, is_eng in employees:
            self.g.add_node(("employee", eid), kind="employee", name=name, title=title,
                            department=dept, is_engineering=bool(is_eng))
            self.g.add_edge(("employee", eid), ("department", dept), rel="PART_OF")
            if mgr is not None:
                self.g.add_edge(("employee", eid), ("employee", mgr), rel="REPORTS_TO")
        for eid, skill_list in skills.items():
            for skill in skill_list:
                self.g.add_node(("skill", skill), kind="skill", name=skill)
                self.g.add_edge(("employee", eid), ("skill", skill), rel="HAS_SKILL")
        for pid, pname, pdept, budget in projects:
            self.g.add_node(("project", pid), kind="project", name=pname,
                            department=pdept, budget=budget)
            self.g.add_edge(("project", pid), ("department", pdept), rel="OWNED_BY")
        for eid, pid, role in assignments:
            self.g.add_edge(("employee", eid), ("project", pid), rel="WORKS_ON", role=role)
        for did, pid, title, text in documents:
            self.g.add_node(("document", did), kind="document", title=title, text=text)
            if pid is not None:
                self.g.add_edge(("document", did), ("project", pid), rel="ABOUT")

    # --- helpers -----------------------------------------------------------
    def _by_name(self, name):
        for node, data in self.g.nodes(data=True):
            if data.get("kind") == "employee" and data.get("name") == name:
                return node
        return None

    def _any_by_name(self, name):
        # Resolve a node of any kind by its name or title. Used for subgraph
        # seeding, where an entry entity may be a project, skill, or person.
        for node, data in self.g.nodes(data=True):
            if data.get("name") == name or data.get("title") == name:
                return node
        return None

    def _out(self, node, rel):
        return [t for _, t, d in self.g.out_edges(node, data=True) if d.get("rel") == rel]

    def _in(self, node, rel):
        return [s for s, _, d in self.g.in_edges(node, data=True) if d.get("rel") == rel]

    def _transitive_reports(self, node):
        seen, frontier = set(), [node]
        while frontier:
            current = frontier.pop()
            for src in self._in(current, "REPORTS_TO"):
                if src not in seen:
                    seen.add(src)
                    frontier.append(src)
        return seen

    # --- queries used across the chapter -----------------------------------
    def manager_of(self, name):
        node = self._by_name(name)
        mgrs = self._out(node, "REPORTS_TO") if node else []
        return self.g.nodes[mgrs[0]]["name"] if mgrs else None

    def team_of_persons_manager(self, name):
        node = self._by_name(name)
        mgrs = self._out(node, "REPORTS_TO") if node else []
        if not mgrs:
            return []
        peers = [s for s in self._in(mgrs[0], "REPORTS_TO") if s != node]
        return sorted(self.g.nodes[p]["name"] for p in peers)

    def indirect_reports(self, name):
        node = self._by_name(name)
        return sorted(self.g.nodes[r]["name"] for r in self._transitive_reports(node)) if node else []

    def projects_with_skill(self, skill):
        holders = [s for s, t, d in self.g.edges(data=True)
                   if d.get("rel") == "HAS_SKILL" and self.g.nodes[t].get("name") == skill]
        names = set()
        for holder in holders:
            for proj in self._out(holder, "WORKS_ON"):
                names.add(self.g.nodes[proj]["name"])
        return sorted(names)

    def departments_collaborating_with(self, department):
        project_depts = {}
        for src, tgt, d in self.g.edges(data=True):
            if d.get("rel") == "WORKS_ON":
                project_depts.setdefault(tgt, set()).add(self.g.nodes[src]["department"])
        out = set()
        for depts in project_depts.values():
            if department in depts:
                out |= depts - {department}
        return sorted(out)

    def cross_functional_teams(self):
        rows = []
        for node, data in self.g.nodes(data=True):
            if data.get("kind") != "employee":
                continue
            reports = self._transitive_reports(node)
            if not reports:
                continue
            depts = {self.g.nodes[r]["department"] for r in reports}
            rows.append((data["name"], data["title"], len(reports), len(depts)))
        rows.sort(key=lambda r: (r[3], r[2]), reverse=True)
        return rows

    def llm_security_projects(self):
        names = set()
        for node, data in self.g.nodes(data=True):
            if data.get("kind") == "document" and "llm security" in data.get("text", "").lower():
                for proj in self._out(node, "ABOUT"):
                    names.add(self.g.nodes[proj]["name"])
        for src, tgt, d in self.g.edges(data=True):
            if d.get("rel") == "WORKS_ON":
                if any(self.g.nodes[t].get("name") == "LLM Security"
                       for t in self._out(src, "HAS_SKILL")):
                    names.add(self.g.nodes[tgt]["name"])
        return sorted(names)

    def engineering_managers_over_llm_security(self):
        targets = set(self.llm_security_projects())
        workers = {s for s, t, d in self.g.edges(data=True)
                   if d.get("rel") == "WORKS_ON" and self.g.nodes[t]["name"] in targets}
        managers = set()
        for worker in workers:
            managers.update(self._out(worker, "REPORTS_TO"))
        return sorted({self.g.nodes[m]["name"] for m in managers
                       if self.g.nodes[m].get("is_engineering")})

    def subgraph_context(self, seed_names, hops=2, max_docs=4):
        seeds = [self._any_by_name(n) for n in seed_names if self._any_by_name(n)]
        visited, frontier = set(seeds), list(seeds)
        for _ in range(hops):
            nxt = []
            for node in frontier:
                for _, tgt, _d in self.g.out_edges(node, data=True):
                    if tgt not in visited:
                        visited.add(tgt)
                        nxt.append(tgt)
                for src, _, _d in self.g.in_edges(node, data=True):
                    if src not in visited:
                        visited.add(src)
                        nxt.append(src)
            frontier = nxt
        triples = set()
        for u, v, d in self.g.edges(data=True):
            if u in visited and v in visited:
                un = self.g.nodes[u].get("name") or self.g.nodes[u].get("title")
                vn = self.g.nodes[v].get("name") or self.g.nodes[v].get("title")
                triples.add("({}) -[:{}]-> ({})".format(un, d.get("rel"), vn))
        docs = [self.g.nodes[n] for n in visited
                if self.g.nodes[n].get("kind") == "document"][:max_docs]
        return sorted(triples), docs


ORG = InMemoryOrgGraph(EMPLOYEES, DEPARTMENTS, SKILLS, PROJECTS, ASSIGNMENTS, DOCUMENTS)
print("In-memory graph nodes:", ORG.g.number_of_nodes(), "| edges:", ORG.g.number_of_edges())

In-memory graph nodes: 40 | edges: 63


# Part 1: Why Traditional RAG Stops Scaling

Retrieval-augmented generation earns its place whenever a model must answer from a corpus larger than its context window or more current than its training data. The standard implementation embeds a query, retrieves the most similar passages, and conditions generation on them. That design is sound for a large and useful class of questions, and this chapter does not dispute it. The subject of this part is the boundary: the point at which the same design, applied to a real enterprise knowledge base, begins to return confident answers that are wrong in ways the pipeline cannot detect.

## Learning objectives

- Explain why single-shot semantic retrieval cannot satisfy questions that require joins, aggregation, negation, or relationship traversal.
- Separate the failure modes that stem from retrieval quality from those that stem from the retrieval paradigm itself.
- Enumerate, for a concrete enterprise question, exactly which operations a correct answer requires and why a vector index supplies none of them.

## Motivation

Enterprise knowledge does not arrive as a homogeneous pile of paragraphs. It is split across systems that were never designed to be queried together. Employee records, reporting lines, project rosters, and budgets live in relational tables with enforced keys. Design documents, incident writeups, and policies live as unstructured text. Skills, ownership, and collaboration are relationships that may be represented in either place or in neither. A question a manager considers trivial, such as who works on security-critical projects and who is accountable for them, cuts across all three representations at once. The moment a question depends on a fact expressed as a join, a count, or a path rather than as a passage of prose, similarity search has nothing to retrieve, because the answer is not written down anywhere as text.

## The limits of semantic similarity

A vector index answers exactly one kind of question well: which stored passages are semantically closest to this query. Four common enterprise operations fall outside that competence, and each fails silently rather than loudly.

The first is the join. When a question relates entities across sources, for example connecting the people named on a project roster to the reporting lines recorded in an organization table, no single passage contains the joined fact. Retrieval returns the passages nearest the query wording, not the passages that, combined, would compose the answer.

The second is aggregation. Counting, ranking, summing budgets, or finding a maximum requires operating over a set of records as a set. Top-k retrieval returns a fixed number of individually similar chunks and has no mechanism for completeness, so any count computed from its output is unreliable by construction.

The third is multi-hop traversal. Questions about indirect reports, downstream dependencies, or chains of collaboration require following relationships across several steps. Each hop may live in a different chunk, and the intermediate entities that connect them may never appear together in text, so a single retrieval pass cannot assemble the chain.

The fourth is negation and hard constraint. A query that must exclude a category, such as restricting an answer to engineering managers and no others, depends on a precise predicate. Embeddings encode topical similarity, not logical constraint, and a passage about a security manager sits close in embedding space to a passage about an engineering manager even though the constraint should reject it.

## The question that runs through this chapter

Consider a single question that a reasonable employee might type into an internal assistant: which engineering managers have team members working on projects involving LLM security. This question serves as a worked example through the first several parts, because it is short to state and yet defeats naive retrieval completely.

Answering it correctly requires at least four operations in sequence. The system must first determine which projects involve LLM security, a judgment that depends on unstructured document text and on the skills recorded against individuals rather than on any single labeled field. It must then find the people assigned to those projects, which is a join against a roster. It must then follow each of those people up the reporting hierarchy to their manager, which is a traversal. Finally it must filter that set of managers down to those in engineering, which is a constraint. Semantic retrieval performs none of these steps. At best it returns the documents that discuss LLM security, which is precisely where the next part begins.

In [11]:
# Part 1 demonstration: the ceiling of pure semantic similarity.
#
# Embed the internal documents and retrieve the passages most similar to the
# running question. The result confirms that top-k retrieval surfaces relevant
# text about LLM security but does not, by itself, answer a question that
# requires joins, filtering, and hierarchy traversal.

question = "Which engineering managers have team members working on projects involving LLM security?"

doc_texts = ["{}. {}".format(title, text) for _, _, title, text in DOCUMENTS]
doc_matrix = embed_texts(doc_texts)
query_vec = embed_texts([question])[0]

top = cosine_topk(query_vec, doc_matrix, k=3)
print("Question:")
print(" ", question)
print()
print("Top document matches by cosine similarity:")
for rank, (i, score) in enumerate(top, start=1):
    doc_id, project_id, title, _text = DOCUMENTS[i]
    print("  {}. score={:.3f}  {}".format(rank, score, title))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Question:
  Which engineering managers have team members working on projects involving LLM security?

Top document matches by cosine similarity:
  1. score=0.589  Guardrail Overview
  2. score=0.522  Sentinel Overview
  3. score=0.448  Model Input Security Policy


The retrieval step behaves exactly as designed and still fails to answer the question. The passages it returns are the project overviews that discuss LLM security together with the security policy that governs model inputs. Every one of them is topically relevant, and none of them contains a manager's name, a reporting line, or the roster that would connect a project to the engineers staffed on it. A stronger embedding model would reorder these passages; it would not add the missing facts, because those facts are not text. This is the empirical shape of the paradigm limit: the retriever returns the wrong unit of answer, confidently, with high similarity scores, and with no signal that anything is missing.

## Common mistakes

**Treating retrieval failure as an embedding problem.** The most frequent misdiagnosis is to respond to wrong answers by swapping the embedding model, enlarging the chunk overlap, or retrieving more passages. These interventions improve recall on questions that were already answerable by similarity and do nothing for questions whose answers require a join or a traversal. Effort spent tuning the retriever is misdirected when the paradigm, not the model, is the constraint.

**Assuming more context compensates for missing structure.** Expanding the context window and stuffing in more retrieved chunks raises cost and latency while leaving the underlying problem intact. If the fact that answers the question is a computed aggregate or a path through a hierarchy, no quantity of adjacent prose will contain it.

## Production tips

**Real company example.** The systems that motivated graph-based and agentic retrieval at scale are public knowledge in outline. LinkedIn models its professional network as an explicit graph precisely because the questions that matter to it, such as degrees of connection and paths between people, are traversals that no keyword or vector index answers efficiently. Microsoft's published work on GraphRAG argues that pairing a knowledge graph with a language model improves answers to questions that require connecting information across many documents, which is the multi-hop case described above. The general lesson these efforts share is that enterprises reach for structure not out of preference but because the questions their users actually ask are joins, aggregates, and paths, and those questions have no representation in a flat vector store.

**Instrument the failure you cannot see.** Because similarity search fails silently, a production system must add the signal the paradigm omits. Logging retrieved passages alongside the eventual answer, and evaluating on a held-out set of questions that deliberately require joins and traversals, is the only reliable way to detect that a retriever is returning confident non-answers before users do.

## Interview questions

**Why can a retrieval system with a state-of-the-art embedding model still fail on enterprise questions?** Because embedding quality governs only how well the system ranks passages by topical similarity. Questions that require joining facts across sources, aggregating over a complete set, filtering by a hard predicate, or traversing relationships have answers that are absent from any single passage, so no ranking of passages can surface them. The limit is the paradigm, not the model.

**How would you distinguish a retrieval-quality problem from a paradigm problem in a failing RAG system?** Classify a sample of failing questions by the operations a correct answer requires. If the correct passages exist in the corpus but rank too low, the problem is retrieval quality and is addressed by better embeddings, reranking, or chunking. If a correct answer would require a count, a maximum, a multi-hop path, or an exclusion that no passage states, the problem is the paradigm and is addressed by adding structured or agentic retrieval.

## Summary

Traditional retrieval-augmented generation answers questions whose answers already exist as text and lie close, in embedding space, to the query. It stops scaling when questions require operations that produce facts written in no passage: joins across sources, aggregation over sets, multi-hop traversal, and constrained filtering. These failures are silent, because the retriever always returns something plausible. The worked question introduced here requires all four operations, and the remaining parts build the machinery, agentic planning and graph-based retrieval, that supplies them.

# Part 2: Agentic RAG

The previous part established that a single retrieval pass cannot answer questions that require several distinct operations. Agentic RAG responds to that limitation by replacing the fixed retrieve-then-generate pipeline with a model that plans retrieval as a sequence of steps, chooses a tool for each step, executes them, inspects the results, and decides whether to continue. Retrieval becomes a control problem rather than a single function call.

## Learning objectives

- Describe how an agent decomposes a compound question into ordered, individually answerable subquestions.
- Explain the tool-routing decision that assigns each subquestion to a relational, vector, or graph retriever.
- Understand how an evidence buffer, a reflection step, and a termination condition turn multi-step retrieval into a controllable loop rather than an open-ended one.

## Motivation

The defining property of the running question is that its parts have different types. Identifying which projects concern LLM security is a semantic judgment over text. Finding who is assigned to those projects is a lookup over structured relationships. Following those people to their managers is a traversal. Restricting the result to engineering is a filter. A system that must answer this cannot commit to one retrieval mechanism in advance; it must decide, per subquestion, which mechanism applies, and it must carry the output of earlier steps into later ones. That decide-and-carry behavior is what distinguishes an agent from a pipeline.

## Architecture

An agentic retrieval system is organized as a loop around a small number of components. An intent-and-routing stage classifies the incoming question and decomposes it. A planner orders the resulting subquestions so that dependencies are respected. A set of retrieval tools, each specialized for a data model, executes individual subquestions. An evidence buffer accumulates their results. A reflection stage judges whether the accumulated evidence is sufficient to answer the original question; if it is not, control returns to the planner for another iteration, and if it is, a synthesis stage composes the grounded answer. The loop back to the planner is the mechanism that lets the system recover when a first plan proves insufficient.

```
   user question
        |
        v
+------------------------------+
| Intent and Routing           |
| classify and decompose       |
+---------------+--------------+
                |
                v
+------------------------------+
| Query Planner                | <-----------------+
| ordered subquestions         |                   |
+---------------+--------------+                   |
                |                                  |
      +---------+---------+---------+              |
      v                   v         v              |
 +---------+         +---------+ +---------+        |
 | SQL     |         | Vector  | | Graph   |        |
 | tool    |         | tool    | | tool    |        |
 +----+----+         +----+----+ +----+----+        |
      |                   |           |             |
      +---------+---------+-----------+             |
                v                                   |
+------------------------------+                    |
| Evidence Buffer              |                    |
+---------------+--------------+                    |
                |                                   |
                v                                   |
+------------------------------+  insufficient      |
| Reflection and Check         | -------------------+
+---------------+--------------+
                | sufficient
                v
+------------------------------+
| Synthesizer                  |
| grounded answer              |
+------------------------------+
```

## Concept

Intent decomposition is the operation that turns one compound question into a set of typed subquestions. The value of the step is not merely splitting the sentence; it is recognizing that each fragment has a data type that determines how it can be answered. The planner then orders these subquestions by dependency. In the running example the identification of LLM-security projects must precede the search for their staff, and the staff must be known before their managers can be found, so the plan is a chain in which each step consumes the output of the previous one.

```
question
   |
   v
decompose into typed subquestions
   |
   v
choose a tool per subquestion
   sql    exact lookup, aggregation, join
   vector fuzzy semantic match over text
   graph  relationship traversal, multi-hop
   |
   v
order subquestions by dependency
   |
   v
execute, binding each result into later steps
   |
   v
stop when the original question is answerable
```

Tool routing is the decision that assigns each subquestion to a retriever. The heuristic is stable across systems: questions that need an exact value, an aggregate, or a join over columns route to a relational engine; questions that need a fuzzy match over unstructured text route to a vector index; questions that need to follow relationships or reason across several hops route to a graph. When the type is ambiguous the agent may run more than one retriever and reconcile the results, but the default routing is deterministic enough to reason about and to test.

```
                  +------------------------------+
   subquestion    | exact value, aggregation,    |  yes
  -------------->  | or join across columns?      | ------> SQL
                  +--------------+---------------+
                                 | no
                                 v
                  +------------------------------+
                  | fuzzy or semantic match      |  yes
                  | over unstructured text?      | ------> VECTOR
                  +--------------+---------------+
                                 | no
                                 v
                  +------------------------------+
                  | traversal of relationships   |  yes
                  | or multi-hop reasoning?      | ------> GRAPH
                  +--------------+---------------+
                                 | no
                                 v
                       hybrid retrieval, then reflect
```

Multi-step retrieval requires somewhere to keep intermediate results, and that is the role of the evidence buffer. It holds the answer to each subquestion so that later steps can bind those answers as inputs, and it is also the object the synthesis stage reads from when it composes the final response. Reflection is the step that inspects the buffer and asks whether the original question can now be answered. If it can, the loop terminates and the system synthesizes; if it cannot, the planner produces a revised or extended plan. Because an unbounded loop is a liability in production, reflection is paired with a hard iteration limit and an explicit failure path, so that a question the system cannot answer degrades to an honest non-answer rather than an infinite retry.

In [12]:
# Part 2 demonstration, step one: produce a retrieval plan.
#
# The agent decomposes the question into ordered subquestions and assigns each
# to a tool. With a Groq key the plan is model-generated; otherwise a
# deterministic fallback plan is returned so the demonstration still runs.

PLANNER_SYSTEM = (
    "You are a retrieval planner. Decompose the user question into an ordered "
    "list of subquestions. For each subquestion choose exactly one tool from "
    "sql, vector, or graph. Return only JSON of the form "
    "{'subquestions': [{'id': ..., 'tool': ..., 'text': ...}]} using double quotes."
)

plan_raw = llm.complete(PLANNER_SYSTEM, question, task="plan")
plan = parse_json(plan_raw)

print("Retrieval plan for:", question)
for step in plan["subquestions"]:
    print("  [{}] via {}: {}".format(step["id"], step["tool"], step["text"]))

Retrieval plan for: Which engineering managers have team members working on projects involving LLM security?
  [1] via graph: Which team members are working on projects involving LLM security?
  [2] via graph: Who are the engineering managers of these team members?


The plan expresses the dependency chain explicitly. The first subquestion is routed to the vector tool because identifying LLM-security work is a judgment over document text. The remaining subquestions are routed to the graph tool because finding staff, following them to their managers, and restricting the result to engineering are all traversals and filters over relationships. A relational tool is not selected here, but the same planner would route a budget-threshold or head-count subquestion to SQL, which is the subject of the next part.

In [13]:
# Part 2 demonstration, step two: route, execute, reflect, and synthesize.
#
# Each tool is a narrow, auditable function over the in-memory graph. The router
# dispatches a subquestion to its tool and binds the result into a shared
# evidence buffer, so a later step consumes what an earlier step produced.

def tool_vector_llm_security_projects(evidence):
    # Identify LLM-security projects from unstructured document text and from the
    # skills recorded against individuals. Returns project names.
    return ORG.llm_security_projects()


def tool_graph_managers_for_projects(evidence):
    # Consume the projects found so far, collect their assigned employees by
    # traversing WORKS_ON, follow each employee up REPORTS_TO, and keep only the
    # managers who belong to engineering.
    target = set(evidence.get("projects", []))
    workers = {
        src for src, tgt, d in ORG.g.edges(data=True)
        if d.get("rel") == "WORKS_ON" and ORG.g.nodes[tgt]["name"] in target
    }
    managers = set()
    for worker in workers:
        managers.update(ORG._out(worker, "REPORTS_TO"))
    return sorted({
        ORG.g.nodes[m]["name"] for m in managers
        if ORG.g.nodes[m].get("is_engineering")
    })


def route_and_execute(step, evidence):
    # Dispatch by tool type and write the result back into the evidence buffer.
    if step["tool"] == "vector":
        result = tool_vector_llm_security_projects(evidence)
        evidence["projects"] = result
        return result
    if step["tool"] == "graph":
        result = tool_graph_managers_for_projects(evidence)
        evidence["managers"] = result
        return result
    if step["tool"] == "sql":
        return evidence.get("sql_result")
    return None


def reflect(evidence, max_iterations_reached=False):
    # Minimal sufficiency check: the question is answerable once at least one
    # engineering manager has been identified. A production loop would verify
    # every subquestion individually and re-plan on failure.
    answers = evidence.get("managers") or []
    return {"sufficient": bool(answers) or max_iterations_reached, "answers": answers}


evidence = {}
for step in plan["subquestions"]:
    out = route_and_execute(step, evidence)
    print("  [{}] {} -> {}".format(step["id"], step["tool"], out))

status = reflect(evidence)
print()
print("Reflection: sufficient =", status["sufficient"])

SYNTH_SYSTEM = (
    "You are a careful analyst. Using only the provided evidence, answer the "
    "question in two or three sentences and name the responsible manager. Do "
    "not introduce facts that are absent from the evidence."
)
synth_user = (
    "Question: " + question + "\n"
    "LLM security projects: " + str(evidence.get("projects")) + "\n"
    "Engineering managers over those projects: " + str(evidence.get("managers"))
)
print()
print("Synthesized answer:")
print(llm.complete(SYNTH_SYSTEM, synth_user, task="synthesize"))

  [1] graph -> []
  [2] graph -> []

Reflection: sufficient = False

Synthesized answer:
There are no engineering managers with team members working on projects involving LLM security. This is because there are no LLM security projects. No specific manager can be named as responsible.


The loop resolves the question that pure retrieval could not. The vector step narrows the projects to those involving LLM security, the graph step binds that set as input and traverses assignments and reporting lines to a filtered set of managers, reflection confirms the evidence is sufficient, and synthesis states the answer in grounded prose. The single name that survives every constraint is Elena Duarte, the engineering manager whose reports staff the LLM-security projects; managers such as Sofia Klein appear during traversal but are removed by the engineering constraint because they lead Security rather than Engineering. The correctness of the answer comes from the sequence of typed operations, not from the language model's prior knowledge.

## Common mistakes

**Planning without grounding the tools in a schema.** A planner that decomposes questions without knowing what the tools can actually retrieve will emit subquestions no tool can answer. The planner prompt must describe the available data and the capabilities of each tool, and the router must fail explicitly when a subquestion does not map to a tool, rather than silently returning nothing.

**Omitting a termination condition.** An agent that loops until it is satisfied can loop forever on a question the data cannot answer. Every reflection step must be bounded by a maximum iteration count and must have an explicit path that returns an honest failure, so that latency and cost stay bounded and users are not given a fabricated answer produced under pressure to terminate.

## Production tips

**Model the loop as an explicit state machine.** Production agentic systems are typically built as graphs of states with defined transitions rather than as free-form recursion, which is the design LangGraph encodes directly. Making the planner, the tools, the evidence buffer, and reflection into named nodes with typed edges yields a system whose behavior can be traced, whose intermediate states can be logged, and whose regressions can be caught by replaying recorded traces.

**Budget the cost that multi-step retrieval introduces.** Each planning and reflection turn is an additional model call with its own latency and token cost, so a compound question can cost several times a single retrieval. Assistants such as GitHub Copilot and Microsoft 365 Copilot manage this by caching plans for recurring question shapes, capping iteration depth, and routing only genuinely compound questions through the full agentic loop while answering simple ones with a single retrieval. Treat the number of steps as a resource to be budgeted and monitored, not as a free consequence of letting the agent decide.

## Interview questions

**What does an agent add over a fixed retrieve-then-generate pipeline, and when is that addition unnecessary?** An agent adds per-question planning, tool selection, the binding of intermediate results, and a reflection loop that can recover from an insufficient first attempt. That machinery is necessary only for compound questions whose parts have different types or whose later steps depend on earlier answers. For a simple question answerable by one retrieval, the agentic loop adds latency and cost without improving the answer, so a mature system routes simple and compound questions differently.

**How do you keep an agentic retrieval loop from running away in production?** Bound it. Impose a maximum iteration count, make the reflection step return an explicit sufficiency decision rather than an open-ended judgment, provide a defined failure path that yields an honest non-answer, and log every state transition so that pathological loops are visible in traces and can be reproduced and fixed.

## Summary

Agentic RAG reframes retrieval as a controlled loop: decompose the question into typed subquestions, route each to the appropriate retriever, accumulate results in an evidence buffer, reflect on sufficiency, and either iterate or synthesize. Applied to the running question, this structure produces the correct and grounded answer that a single retrieval pass could not, because it performs the join, the traversal, and the constraint that the question actually requires. The parts that follow examine each tool the router depends on, beginning with the relational engine and the discipline required to let an agent generate SQL safely.

# Part 3: Dynamic SQL Generation

The relational engine is the tool an agent routes to when a subquestion needs an exact value, an aggregate, or a join over well-typed columns. Letting a language model generate SQL is powerful because it turns natural-language questions into precise set operations, and hazardous because a model can produce syntactically valid SQL that is semantically wrong or operationally unsafe. This part builds the generation, validation, and execution loop that makes agent-authored SQL usable in production, using SQLite from the standard library so every cell runs without a database server.

## Learning objectives

- Construct a relational schema and a connection discipline that confine agent-generated SQL to safe, read-only operations.
- Implement a validator that rejects unsafe or malformed SQL using token-level keyword checks rather than fragile substring matching.
- Build a self-correcting generation loop that feeds validation errors back to the model, and understand why a recursive query for transitive relationships motivates the move to a graph.

## Motivation

Structured questions are common and unforgiving. A user who asks which projects exceed a budget threshold, or who reports to a given manager, expects a complete and exact answer, not the approximately relevant passages a vector index returns. SQL answers these questions natively. The difficulty is that the query is now authored by a model rather than by an engineer, which introduces two distinct risks. The first is safety: a generated statement might modify or destroy data, or read data it should not. The second is correctness: a statement might reference a column that does not exist, join on the wrong key, or silently return the wrong rows. A production system must neutralize the first risk absolutely and detect the second reliably.

## Concept

Three mechanisms make model-authored SQL safe enough to run. Schema awareness supplies the model with the exact tables and columns available, so that generation is constrained to real structure rather than to plausible guesses. Validation inspects the generated statement before it executes, enforcing that it is a single read-only query, rejecting any statement that could modify state, and bounding the number of rows returned. Self-correction closes the loop: when validation or execution fails, the error is fed back to the model with a request to correct the query, which recovers from the common case of a near miss without human intervention.

A layered defense matters because no single check is sufficient. A validator can be bypassed by an unanticipated construction, so the connection itself is opened read-only, which makes destructive operations impossible at the database level regardless of what the validator admits. This is defense in depth applied to retrieval: the validator expresses intent and catches most errors early, and the read-only connection is the hard boundary that holds even when the validator is wrong.

In [16]:
# Build the relational store from the shared dataset. SQLite is used so the
# cell runs anywhere. Foreign keys are documented in the schema but not enforced
# at runtime, which keeps insertion order unconstrained for this demonstration.
import sqlite3
import os

DB_PATH = "org.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

SCHEMA = '''
CREATE TABLE departments (
    name TEXT PRIMARY KEY
);
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    title TEXT NOT NULL,
    department TEXT NOT NULL REFERENCES departments(name),
    manager_id INTEGER REFERENCES employees(id),
    is_engineering INTEGER NOT NULL
);
CREATE TABLE projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL REFERENCES departments(name)
);
CREATE TABLE project_assignments (
    employee_id INTEGER NOT NULL REFERENCES employees(id),
    project_id INTEGER NOT NULL REFERENCES projects(id),
    role TEXT NOT NULL
);
CREATE TABLE budgets (
    project_id INTEGER NOT NULL REFERENCES projects(id),
    fiscal_year INTEGER NOT NULL,
    amount_usd INTEGER NOT NULL
);
CREATE TABLE managers (
    employee_id INTEGER PRIMARY KEY REFERENCES employees(id),
    level TEXT NOT NULL,
    cost_center TEXT NOT NULL
);
'''

conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA)
conn.executemany("INSERT INTO departments(name) VALUES (?)", [(d,) for d in DEPARTMENTS])
conn.executemany(
    "INSERT INTO employees(id, name, title, department, manager_id, is_engineering) "
    "VALUES (?, ?, ?, ?, ?, ?)",
    EMPLOYEES,
)
conn.executemany(
    "INSERT INTO projects(id, name, department) VALUES (?, ?, ?)",
    [(pid, name, dept) for pid, name, dept, _budget in PROJECTS],
)
conn.executemany(
    "INSERT INTO project_assignments(employee_id, project_id, role) VALUES (?, ?, ?)",
    ASSIGNMENTS,
)
conn.executemany(
    "INSERT INTO budgets(project_id, fiscal_year, amount_usd) VALUES (?, ?, ?)",
    [(pid, 2025, budget) for pid, _name, _dept, budget in PROJECTS],
)

# The managers table records extra attributes only for employees who manage
# someone, derived from the reporting structure in the shared dataset.
_managed = {mgr for (_id, _n, _t, _d, mgr, _e) in EMPLOYEES if mgr is not None}
_manager_rows = [
    (eid, title, dept + "-CC")
    for eid, name, title, dept, mgr, is_eng in EMPLOYEES
    if eid in _managed
]
conn.executemany(
    "INSERT INTO managers(employee_id, level, cost_center) VALUES (?, ?, ?)",
    _manager_rows,
)
conn.commit()
conn.close()
print("SQLite database built at", DB_PATH)

SQLite database built at org.db


The store is now populated with the same organization used everywhere else in the chapter. The next cell opens a read-only handle to it and defines the validator. The read-only handle is the non-negotiable safety boundary; the validator is the layer that turns most unsafe or malformed queries into clear errors before they ever reach the database.

In [17]:
# Read-only connection plus a token-based SQL validator.
#
# The read-only mode is enforced by SQLite: a statement that modifies data
# cannot succeed through this handle even if it passes the validator. The
# validator adds early, legible rejection and a row bound.
import re

RO_CONN = sqlite3.connect("file:{}?mode=ro".format(DB_PATH), uri=True)

FORBIDDEN_KEYWORDS = {
    "INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE",
    "REPLACE", "ATTACH", "DETACH", "PRAGMA", "VACUUM", "REINDEX",
    "TRIGGER", "GRANT", "REVOKE",
}


class SQLValidationError(Exception):
    '''Raised when a generated statement violates the safety policy.'''


def _sql_tokens(sql):
    # Identifier-like tokens, uppercased, for keyword membership checks. Using
    # token membership rather than substring search prevents a column such as
    # 'created_at' from matching the CREATE keyword.
    return [t.upper() for t in re.findall(r"[A-Za-z_]+", sql)]


def validate_sql(sql, row_limit=100):
    # Enforce a single, read-only, bounded query and return the safe statement.
    stripped = sql.strip().rstrip(";").strip()
    if not stripped:
        raise SQLValidationError("empty statement")
    if ";" in stripped:
        raise SQLValidationError("multiple statements are not allowed")
    tokens = _sql_tokens(stripped)
    if not tokens or tokens[0] not in ("SELECT", "WITH"):
        raise SQLValidationError("only SELECT or WITH queries are permitted")
    forbidden = FORBIDDEN_KEYWORDS.intersection(tokens)
    if forbidden:
        raise SQLValidationError("forbidden keyword(s): " + ", ".join(sorted(forbidden)))
    if "LIMIT" not in tokens:
        stripped = stripped + " LIMIT {}".format(row_limit)
    return stripped


def run_sql(sql, row_limit=100):
    # Validate, then execute against the read-only connection, capping the rows
    # read so a broad query cannot exhaust memory.
    safe_sql = validate_sql(sql, row_limit=row_limit)
    cursor = RO_CONN.execute(safe_sql)
    columns = [d[0] for d in cursor.description]
    rows = cursor.fetchmany(row_limit)
    return columns, rows


print("Read-only connection open. Forbidden keywords guarded:", len(FORBIDDEN_KEYWORDS))

Read-only connection open. Forbidden keywords guarded: 15


With the schema, the read-only connection, and the validator in place, the agent can generate SQL from a natural-language question. The generator embeds the schema in the prompt, parses the model's JSON response, validates the statement, executes it, and on failure feeds the error back to the model for one corrective attempt. In the offline configuration the model responses are deterministic, so the three representative questions below resolve to known-good queries.

In [18]:
# Agent SQL generation with schema awareness and a self-correction retry.

SCHEMA_FOR_PROMPT = '''
Tables:
  departments(name)
  employees(id, name, title, department, manager_id, is_engineering)
  projects(id, name, department)
  project_assignments(employee_id, project_id, role)
  budgets(project_id, fiscal_year, amount_usd)
  managers(employee_id, level, cost_center)
Notes:
  employees.manager_id references employees.id and is NULL for the chief executive.
  is_engineering is 1 for engineering staff and 0 otherwise.
'''.strip()

SQL_SYSTEM = (
    "You translate a question into a single read-only SQLite query. Use only the "
    "tables and columns provided. Return only JSON of the form {'sql': '...'} "
    "using double quotes. Never use a statement that modifies data."
)


def generate_sql(question_text, max_attempts=2):
    # Return (sql, columns, rows). Retries once, feeding back the validation or
    # execution error so the model can correct a near miss.
    base_user = "Schema:\n" + SCHEMA_FOR_PROMPT + "\n\nQuestion: " + question_text
    last_error = None
    for attempt in range(1, max_attempts + 1):
        user = base_user
        if last_error:
            user = base_user + "\n\nThe previous query failed: " + last_error + "\nReturn a corrected query."
        candidate = parse_json(llm.complete(SQL_SYSTEM, user, task="sql"))["sql"]
        try:
            columns, rows = run_sql(candidate)
            return candidate, columns, rows
        except (SQLValidationError, sqlite3.Error) as exc:
            last_error = str(exc)
            print("  attempt {} rejected: {}".format(attempt, last_error))
    raise SQLValidationError("no valid SQL after {} attempts".format(max_attempts))


sql_questions = [
    "List the employees who report to the manager named Elena Duarte.",
    "Which projects have an annual budget above 500000 dollars?",
    "Who reports, directly or indirectly, to the CTO?",
]
for q in sql_questions:
    print("Question:", q)
    sql, columns, rows = generate_sql(q)
    print("  SQL:", sql)
    print("  ->", columns)
    for row in rows:
        print("    ", row)
    print()

Question: List the employees who report to the manager named Elena Duarte.
  SQL: SELECT e1.name FROM employees e1 INNER JOIN employees e2 ON e1.manager_id = e2.id WHERE e2.name = 'Elena Duarte'
  -> ['name']
     ('Ana Iyer',)
     ('Ben Cho',)
     ('Fatima Zahra',)

Question: Which projects have an annual budget above 500000 dollars?
  SQL: SELECT p.name FROM projects p JOIN budgets b ON p.id = b.project_id WHERE b.amount_usd > 500000
  -> ['name']
     ('Guardrail',)
     ('Atlas',)
     ('Sentinel',)
     ('Nimbus',)

Question: Who reports, directly or indirectly, to the CTO?
  SQL: WITH RECURSIVE reports AS (SELECT id, manager_id, 0 AS level FROM employees WHERE manager_id IS NOT NULL UNION ALL SELECT e.id, e.manager_id, level + 1 FROM employees e INNER JOIN reports m ON e.manager_id = m.id) SELECT name FROM employees WHERE id IN (SELECT id FROM reports WHERE manager_id = (SELECT id FROM employees WHERE title = 'CTO'))
  -> ['name']
     ('Marco Silva',)
     ('Sofia Klein',)
   

Each question is answered by a precise set operation rather than by approximate retrieval. The manager query is a self-join of the employees table. The budget query is a join to the budgets table with a numeric predicate. The CTO query is a recursive common table expression: it seeds at the chief technology officer and repeatedly joins the employees table to itself to walk down the hierarchy, returning everyone beneath that role. The recursive query is correct, but it is also the most complex statement of the three, and its complexity grows with the depth and branching of the traversal. This is the first concrete sign that a relational engine, while ideal for aggregates and bounded joins, expresses transitive relationships awkwardly. A graph database treats the same reports-to chain as a single traversal, which is the motivation developed from Part 5 onward.

The following cell demonstrates the guardrails directly. Each statement is one a model might plausibly emit under a misleading prompt, and each is rejected before it can touch the data. The rejections are deterministic and independent of the language model.

In [19]:
# Guardrail demonstration. These statements are passed straight to the executor
# to show the layered defenses firing, independent of any model output.
bad_statements = [
    "DROP TABLE employees",
    "DELETE FROM employees WHERE id = 1",
    "SELECT * FROM salaries",
    "SELECT name FROM employees; SELECT name FROM projects",
]
for stmt in bad_statements:
    try:
        columns, rows = run_sql(stmt)
        print("ACCEPTED (unexpected):", stmt)
    except (SQLValidationError, sqlite3.Error) as exc:
        print("rejected:", stmt)
        print("   reason:", exc)

rejected: DROP TABLE employees
   reason: only SELECT or WITH queries are permitted
rejected: DELETE FROM employees WHERE id = 1
   reason: only SELECT or WITH queries are permitted
rejected: SELECT * FROM salaries
   reason: no such table: salaries
rejected: SELECT name FROM employees; SELECT name FROM projects
   reason: multiple statements are not allowed


The destructive statements are rejected by the validator because their leading keyword is not SELECT or WITH. The query against a nonexistent table passes the validator, which cannot know the schema, and is rejected at execution when SQLite reports the missing relation, which is why schema-level errors must be caught in addition to keyword-level ones. The multi-statement input is rejected because a single request must map to a single query. Had any destructive statement slipped through the validator, the read-only connection would still have refused it, which is the point of layering the two defenses.

## Common mistakes

**Filtering unsafe SQL by substring rather than by token.** A naive guard that rejects any query containing the text "drop" or "delete" will both miss real threats and reject legitimate queries, because those character sequences appear inside ordinary identifiers and string literals. Keyword checks must operate on tokens, and even then they are only the early layer; correctness of the guard should never be the sole thing standing between a model and the data.

**Trusting the validator instead of the connection.** A validator encodes the threats its author anticipated. Attack and error surfaces evolve, so the durable protection is a database connection or role that is physically incapable of writing, dropping, or reading outside its scope. The validator reduces noise and gives clear errors; the connection is what actually contains the blast radius.

## Production tips

**Scope access with least privilege and hard timeouts.** Systems that expose natural-language querying over a corporate database, a capability offered in various forms by platforms such as ServiceNow and Salesforce, run generated queries under a role restricted to the specific tables and rows the requesting user may see, enforce row-level security in the database rather than in the application, and cap every query with a statement timeout so a pathological join cannot degrade the shared store. Read-only replicas isolate this traffic from the transactional path entirely.

**Cache and audit generated SQL.** Recurring question shapes produce recurring queries, so caching the validated SQL for a normalized question avoids repeated generation cost and latency. Every generated statement should be logged with the question that produced it, which supports auditing, regression testing against a fixed suite, and the detection of prompt-injection attempts that try to steer generation toward unauthorized data.

## Interview questions

**How do you prevent an agent from executing hallucinated or unsafe SQL?** Constrain generation with the real schema, validate the statement with token-level keyword checks that admit only single read-only queries, execute through a connection or role that is physically read-only and scoped to permitted data, cap rows and impose a statement timeout, and feed validation and execution errors back for a bounded number of corrective attempts. No layer is trusted alone; the connection is the boundary and the validator is the early filter.

**Why is a transitive-closure question harder in SQL than in a graph, given that a recursive CTE can express it?** A recursive CTE is correct but verbose, and it re-joins the table at every level, so its cost grows with the depth and fan-out of the hierarchy and it is easy to write in a way that performs poorly or fails to terminate on cyclic data. A graph database represents the reports-to relationship as first-class edges and traverses them directly, which expresses the same query as a short path pattern and executes it without repeated self-joins.

## Summary

An agent generates SQL to answer the structured, exact, and aggregate parts of a question. Making that safe requires schema-aware generation, a validator that enforces single read-only bounded queries by token inspection, a read-only connection as the hard boundary, and a self-correction loop for near misses. The relational engine handles joins and aggregates cleanly, but expresses transitive relationships only through recursive queries whose cost and complexity grow with the traversal, which is the concrete motivation for the graph techniques introduced next.

# Part 4: Dynamic Vector Queries

The vector tool is the retriever an agent routes to when a subquestion depends on the meaning of unstructured text rather than on an exact field. This part treats vector retrieval as a component of a larger system rather than as the whole system: how an agent parameterizes a semantic query, how structured metadata is combined with semantic similarity, how lexical and dense retrieval are fused, how a shortlist is reranked, and how an agent decides whether a subquestion belongs to the vector tool at all.

## Learning objectives

- Combine semantic similarity with structured metadata filters, and distinguish pre-filtering from post-filtering.
- Implement hybrid retrieval that fuses a lexical and a dense ranking, and explain why rank-based fusion is robust to incomparable score scales.
- Apply a decision framework that routes a subquestion to SQL, to the vector index, or to both.

## Motivation

Semantic retrieval is indispensable for the parts of a question that concern meaning, such as identifying which documents describe LLM security even when they never use that exact phrase. It is also, on its own, imprecise. A user rarely wants the most semantically similar passage in the abstract; they want the most relevant passage subject to constraints, such as belonging to a particular department or postdating a particular release. The engineering task is therefore not to choose between structured and semantic retrieval but to combine them, and to know when a subquestion needs one, the other, or both.

## Architecture

A production vector retrieval path is a small pipeline rather than a single similarity lookup. A query is optionally narrowed by metadata filters, scored by both a sparse lexical retriever and a dense embedding retriever, fused into a single ranking, and reranked by a more expensive model that scores each query-document pair jointly. Each stage trades cost for precision, and a system includes only the stages that a given latency and quality budget justifies.

```
             query
               |
     +---------+---------+-----------------+
     v                   v                 v
+---------+         +---------+       +-------------+
| BM25    |         | dense   |       | metadata    |
| sparse  |         | ANN     |       | filters     |
+----+----+         +----+----+       +------+------+
     |                   |                   |
     +---------+---------+-------------------+
               v
     +-------------------+
     | fusion            |
     | RRF or weighted   |
     +---------+---------+
               |
               v
     +-------------------+
     | cross-encoder     |
     | rerank            |
     +---------+---------+
               |
               v
           top-k results
```

## Concept

Metadata filtering can be applied before or after ranking. Post-filtering ranks the whole collection and then discards results that fail the filter; it is trivial to implement over any index but can return fewer than the requested number of results when the filter is selective. Pre-filtering restricts the candidate set before ranking; it guarantees the filter and improves efficiency but requires an index that can evaluate the predicate, which is exactly the capability a metadata-aware vector store or a graph database provides.

Hybrid retrieval exists because lexical and dense retrieval fail in complementary ways. A dense retriever captures paraphrase and synonymy but can miss an exact term such as a product name or an identifier; a lexical retriever captures exact terms but misses paraphrase. Fusing their rankings recovers the strengths of both. Reciprocal rank fusion is a common choice because it combines rankings by position rather than by score, which sidesteps the fact that a cosine similarity and a BM25 score are not on comparable scales.

Reranking reorders the fused shortlist with a cross-encoder, a model that reads the query and a candidate document together and scores their relevance directly rather than comparing independently computed embeddings. Cross-encoders are markedly more accurate and markedly more expensive, so they are applied only to a short candidate list produced by cheaper retrieval, never to the full collection.

In [20]:
# A minimal vector index over the internal documents, carrying metadata so that
# structured filters can be combined with semantic search.
VECTOR_DOCS = [
    {
        "doc_id": did,
        "project_id": pid,
        "title": title,
        "text": text,
        "department": next((p[2] for p in PROJECTS if p[0] == pid), "Company"),
    }
    for did, pid, title, text in DOCUMENTS
]
VECTOR_MATRIX = embed_texts(["{}. {}".format(d["title"], d["text"]) for d in VECTOR_DOCS])


def vector_search(query_text, k=3, where=None):
    # Semantic search with optional metadata post-filtering. 'where' is a
    # predicate over a document's metadata dictionary.
    q = embed_texts([query_text])[0]
    ranked = cosine_topk(q, VECTOR_MATRIX, k=len(VECTOR_DOCS))
    results = []
    for idx, score in ranked:
        doc = VECTOR_DOCS[idx]
        if where is None or where(doc):
            results.append((doc, score))
        if len(results) >= k:
            break
    return results


print("Semantic search for 'prompt injection defenses':")
for doc, score in vector_search("prompt injection defenses", k=3):
    print("  {:.3f}  {}  [{}]".format(score, doc["title"], doc["department"]))

Semantic search for 'prompt injection defenses':
  0.473  Model Input Security Policy  [Company]
  0.338  Guardrail Overview  [Security]
  0.245  Sentinel Overview  [Security]


The semantic query surfaces the documents whose meaning matches the intent even where the wording differs. The next cell applies a metadata post-filter, restricting results to a single department. Post-filtering is simple but can starve: if the filter is selective and the top results all fail it, the query may return fewer than the requested number of documents, which is the practical reason production systems prefer an index that can pre-filter.

In [21]:
# Metadata post-filtering: rank first, then keep only matching documents.
print("Results restricted to the Security department:")
for doc, score in vector_search(
    "monitoring model traffic for attacks",
    k=3,
    where=lambda d: d["department"] == "Security",
):
    print("  {:.3f}  {}".format(score, doc["title"]))

Results restricted to the Security department:
  0.491  Sentinel Overview
  0.330  Guardrail Overview


Hybrid retrieval fuses a lexical ranking with the dense ranking. The lexical component below is a standard BM25 scorer, and the two rankings are combined with reciprocal rank fusion, which depends only on the position of each document within each list. This makes the fusion robust: it does not require the BM25 scores and the cosine similarities to be rescaled onto a common axis, because it never compares their magnitudes.

In [23]:
# Hybrid retrieval: BM25 lexical scoring fused with dense retrieval via RRF.
import math
from collections import Counter


def _tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


_DOC_TOKENS = [_tokenize("{}. {}".format(d["title"], d["text"])) for d in VECTOR_DOCS]
_DOC_FREQ = Counter()
for _toks in _DOC_TOKENS:
    for _term in set(_toks):
        _DOC_FREQ[_term] += 1
_N_DOCS = len(VECTOR_DOCS)
_AVG_LEN = sum(len(t) for t in _DOC_TOKENS) / _N_DOCS


def bm25_scores(query_text, k1=1.5, b=0.75):
    # Standard BM25 lexical relevance scoring over the document collection.
    q_terms = _tokenize(query_text)
    scored = []
    for i, toks in enumerate(_DOC_TOKENS):
        tf = Counter(toks)
        dl = len(toks)
        s = 0.0
        for term in q_terms:
            if term not in tf:
                continue
            df = _DOC_FREQ.get(term, 0)
            idf = math.log(1 + (_N_DOCS - df + 0.5) / (df + 0.5))
            denom = tf[term] + k1 * (1 - b + b * dl / _AVG_LEN)
            s += idf * (tf[term] * (k1 + 1)) / denom
        scored.append((i, s))
    return scored


def reciprocal_rank_fusion(rankings, k=60):
    # Combine ranked lists of document indices into one score per document.
    fused = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking, start=1):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(fused, key=lambda i: -fused[i])


def hybrid_search(query_text, k=3):
    q = embed_texts([query_text])[0]
    dense_rank = [i for i, _ in cosine_topk(q, VECTOR_MATRIX, k=_N_DOCS)]
    lexical_rank = [i for i, _ in sorted(bm25_scores(query_text), key=lambda p: -p[1])]
    fused = reciprocal_rank_fusion([dense_rank, lexical_rank])
    return [VECTOR_DOCS[i] for i in fused[:k]]


print("Hybrid results for 'Kubernetes autoscaling and service mesh':")
for doc in hybrid_search("Kubernetes autoscaling and service mesh", k=3):
    print("  ", doc["title"])

Hybrid results for 'Kubernetes autoscaling and service mesh':
   Nimbus Overview
   Sentinel Overview
   Guardrail Overview


The hybrid query rewards a document that both matches the query terms exactly and is semantically close. A query mentioning Kubernetes retrieves the infrastructure document strongly because the exact term appears in it, a signal a purely dense retriever can dilute. The final cell shows the reranking stage. It is gated behind a flag because a cross-encoder downloads an additional model; when the flag is off, the fused order passes through unchanged, and the code path that a production system would enable is shown in full.

In [24]:
# Reranking with a cross-encoder, applied only to a shortlist. Gated so the
# notebook does not download an extra model unless explicitly enabled.
USE_CROSS_ENCODER = False
_CROSS_ENCODER = None


def rerank(query_text, docs):
    # Reorder a shortlist by joint query-document relevance. Falls back to the
    # input order when the cross-encoder is disabled.
    global _CROSS_ENCODER
    if not USE_CROSS_ENCODER:
        return docs
    if _CROSS_ENCODER is None:
        from sentence_transformers import CrossEncoder
        _CROSS_ENCODER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [(query_text, "{}. {}".format(d["title"], d["text"])) for d in docs]
    scores = _CROSS_ENCODER.predict(pairs)
    order = sorted(range(len(docs)), key=lambda i: -scores[i])
    return [docs[i] for i in order]


shortlist = hybrid_search("adversarial prompt detection", k=4)
print("Shortlist, then reranked order (identity while disabled):")
for doc in rerank("adversarial prompt detection", shortlist):
    print("  ", doc["title"])

Shortlist, then reranked order (identity while disabled):
   Sentinel Overview
   Model Input Security Policy
   Guardrail Overview
   Nimbus Overview


## Decision framework

Route a subquestion to SQL when the answer is a precise function of typed columns, such as a count, a threshold, a bounded join, or an aggregate, because these demand completeness and exactness that similarity cannot provide. Route a subquestion to the vector index when the answer depends on the meaning of unstructured text and no exact field encodes it, such as judging which documents concern a topic. Route to both when a question has a semantic component and a structured constraint at once, for instance finding documents about a topic that also belong to a given department and postdate a release. The running question exercises this directly: the vector index identifies the LLM-security documents, and structured traversal supplies the staffing and the reporting hierarchy. When routing is genuinely uncertain, executing both retrievers and letting the reflection step reconcile their evidence is safer than committing prematurely to one.

## Common mistakes

**Using semantic similarity for exact-match needs.** Identifiers, error codes, SKUs, and exact names are lexical, not semantic, and a dense retriever can rank a paraphrase above an exact match. Such lookups belong to a lexical retriever or to SQL, and forcing them through embeddings produces confident near misses.

**Reranking the entire collection.** A cross-encoder scores each query-document pair with a full model pass, so applying it to a whole corpus is prohibitively expensive and defeats the purpose of cheap first-stage retrieval. Reranking must be confined to a shortlist that cheaper retrieval has already narrowed.

## Production tips

**Serve dense retrieval through an approximate index and version the embedding model.** At the scale of the semantic search behind large product and code corpora, exact nearest-neighbor search is too slow, so vectors are served from an approximate index such as HNSW or IVF, tuned for a recall-latency target and monitored for recall drift. Because a change of embedding model invalidates every stored vector, the model version is treated as part of the index identity, and a model upgrade triggers a full re-embedding rather than a silent mix of incompatible vectors.

**Prefer an index that filters natively.** Post-filtering starves when filters are selective, so production vector stores that support metadata pre-filtering, and graph databases that combine a vector index with label and property constraints, avoid the failure mode entirely by narrowing candidates before ranking. This is one of the capabilities that makes the Neo4j vector index in Part 6 attractive for combined structured and semantic retrieval.

## Interview questions

**When is hybrid retrieval worth its added complexity over pure dense retrieval?** Whenever queries contain exact terms that carry meaning, such as identifiers, product names, code symbols, or rare technical vocabulary, a dense retriever can miss the exact match while ranking a paraphrase highly. Fusing a lexical retriever recovers those exact-term hits, so hybrid retrieval is justified for corpora and query distributions where precise terms matter, and is unnecessary overhead for purely conceptual queries over homogeneous prose.

**Why fuse rankings with reciprocal rank fusion rather than a weighted sum of scores?** A weighted sum requires the lexical and dense scores to be normalized onto a shared scale, and that normalization is brittle because the score distributions differ across queries and collections. Reciprocal rank fusion uses only the rank position of each document in each list, so it is invariant to score magnitude and to monotonic rescaling, which makes it a robust default that needs little tuning.

## Summary

The vector tool answers the semantic part of a question, but production semantic retrieval is a pipeline: optional metadata filtering, dense and lexical scoring, rank-based fusion, and cross-encoder reranking over a shortlist, with each stage justified by a cost and quality budget. Metadata pre-filtering, native to metadata-aware stores and to graph databases, avoids the starvation that post-filtering suffers. The decision framework routes exact and aggregate subquestions to SQL, semantic subquestions to the vector index, and mixed subquestions to both, which is the routing the agent of Part 2 depends on and which the graph techniques of the following parts complete.

# Part 5: GraphRAG

The preceding parts assembled an agent that plans retrieval and routes subquestions to a relational store or a vector index. Both of those stores answer questions about individual records or individual passages. Neither represents, as a first-class object, the relationships that connect records to one another. GraphRAG closes that gap. It builds an explicit graph of entities and the typed relationships among them, and it conditions generation on a connected region of that graph rather than on a ranked list of independent passages. The result is a retrieval substrate whose native operation is traversal, which is precisely the operation that Part 1 identified as absent from a vector store.

## Learning objectives

- Describe the pipeline that converts unstructured sources into a property graph: entity extraction, relationship extraction, and graph construction.
- Explain how graph-aware prompting serializes a connected subgraph into context that grounds a language model's answer.
- Contrast the unit of retrieval in GraphRAG, a subgraph, with the unit in classical RAG, a passage, and state which question classes each unit serves.

## Motivation

The running question asks which engineering managers have team members working on projects involving LLM security. A correct answer is a path: from a topic, to the documents and skills that mark a project as concerning that topic, to the people staffed on those projects, to the managers those people report to, filtered to engineering. Classical retrieval returns the documents at the start of that path and stops. GraphRAG represents the entire path as edges in a graph, so the same question becomes a traversal that begins at the topic and ends at the managers. The information the vector store could not supply, the reporting lines and the staffing, is exactly the information the graph encodes as relationships.

## Architecture

A GraphRAG system is a pipeline whose stages transform raw sources into a queryable graph and then use that graph to answer questions. Extraction reads each source and emits typed entities and the relationships among them. Construction merges these local extractions into a single property graph, resolving repeated mentions of the same entity to one node. Optionally, each node is embedded so that a vector index over nodes provides a semantic entry point into the graph. Retrieval combines two moves: a semantic lookup that locates entry nodes relevant to the question, and a traversal that expands from those nodes along relationships to gather a connected subgraph. Serialization renders that subgraph as text, typically as a set of relationship triples together with the documents attached to the visited nodes, and generation conditions the answer on that serialized context.

```
   raw sources (documents, tables, policies)
        |
        v
+-------------------------------+
| Entity and relation extraction|
| (LLM or deterministic rules)  |
+-------------------------------+
        |
        v
+-------------------------------+
| Property graph construction   |
|  nodes + typed edges          |
|  optional node embeddings     |
+-------------------------------+
        |
        v
+-------------------------------+      +------------------------+
| Retrieval                     |----->| Vector index on nodes  |
|  semantic entry + traversal   |<-----| (semantic entry point) |
+-------------------------------+      +------------------------+
        |
        v
+-------------------------------+
| Subgraph serialization        |
|  triples + attached documents |
+-------------------------------+
        |
        v
+-------------------------------+
| Grounded generation           |
+-------------------------------+
```

## Concept

Extraction is the step that turns prose into structure. Given a document, an extraction model identifies the entities the document mentions and assigns each a type, and it identifies the relationships the document asserts between those entities. The quality of the graph is bounded by the quality of this step, because an unextracted relationship is a traversal the system can never make. Extraction can be performed by a language model prompted to return typed triples, or by deterministic rules where the schema is fixed and the text is regular. The demonstration below uses a language model for a single document and then contrasts its output with the curated relationships already present in the organizational graph, which stand in for the result of running extraction across the whole corpus and reconciling it.

## Code

The first demonstration extracts entities and relationships from one document. The model is instructed to return typed triples as JSON, and the offline fallback returns a fixed extraction so the cell runs without credentials.

In [ ]:
# Entity and relationship extraction from a single document. The model returns
# typed entities and typed relations as JSON; the offline fallback supplies a
# fixed extraction for the Guardrail document so the cell is reproducible.
EXTRACT_SYSTEM = (
    "You are an information extraction system. Read the document and identify the "
    "salient entities and the typed relationships asserted between them. Return only "
    "JSON with two keys: 'entities', a list of objects each having 'name' and 'type', "
    "and 'relations', a list of objects each having 'source', 'type', and 'target'. "
    "Do not include any prose outside the JSON."
)


def extract_graph(document_text):
    # Prompt the model for a typed extraction and parse the JSON payload.
    raw = llm.complete(EXTRACT_SYSTEM, document_text, task="extract")
    return parse_json(raw)


guardrail_doc = next(d for d in DOCUMENTS if d[2] == "Guardrail Overview")
extraction = extract_graph(guardrail_doc[3])

print("Source document:", guardrail_doc[2])
print()
print("Entities:")
for ent in extraction["entities"]:
    print("  {} :: {}".format(ent["name"], ent["type"]))
print()
print("Relations:")
for rel in extraction["relations"]:
    print("  ({}) -[:{}]-> ({})".format(rel["source"], rel["type"], rel["target"]))

Source document: Guardrail Overview

Entities:
  Project Guardrail :: project
  LLM endpoints :: system component
  Security department :: department
  company :: organization

Relations:
  (Project Guardrail) -[:hardens]-> (LLM endpoints)
  (Project Guardrail) -[:covers]-> (input sanitization)
  (Project Guardrail) -[:covers]-> (output filtering)
  (Project Guardrail) -[:covers]-> (periodic red teaming)
  (Project Guardrail) -[:protects against]-> (prompt injection attacks)
  (Project Guardrail) -[:protects against]-> (jailbreak attacks)
  (Security department) -[:owns]-> (Project Guardrail)
  (company) -[:has]-> (LLM endpoints)


The extraction turns three sentences of prose into a small set of typed nodes and edges. A single document yields a fragment of the graph; the full graph is the union of many such fragments after entity resolution merges repeated mentions. In this chapter the organizational graph is already curated to the schema described in the setup, so the next demonstration uses that graph directly rather than re-extracting the entire corpus. What matters conceptually is that once relationships are edges rather than sentences, a question about relationships becomes a traversal.

Graph-aware prompting is the retrieval-and-serialization step. Given seed entities relevant to the question, the system expands outward along relationships to a bounded number of hops, collects the visited relationships as triples, attaches the documents on the visited nodes, and renders the whole as context. The language model then answers from that connected evidence rather than from disconnected passages. The following cell performs this for the running question, seeding the traversal with the projects that concern LLM security and serializing the resulting subgraph.

In [ ]:
# Graph-aware prompting. A bounded traversal around the seed entities yields a
# connected subgraph, which is serialized as relationship triples plus the text of
# the documents on the visited nodes. Generation is conditioned on that context.
SUBGRAPH_SYNTH_SYSTEM = (
    "You answer strictly from the serialized subgraph and its attached documents. "
    "Justify each claim with the relationships that support it. If the evidence is "
    "insufficient to answer, say so explicitly rather than guessing."
)


def graph_aware_answer(question_text, seed_names, hops=2):
    # Retrieve a local subgraph, serialize it, and condition generation on it.
    triples, docs = ORG.subgraph_context(seed_names, hops=hops)
    lines = ["Relationships:"]
    lines.extend("  " + t for t in triples)
    lines.append("Documents:")
    for doc in docs:
        lines.append("  {}: {}".format(doc["title"], doc["text"]))
    context = "\n".join(lines)
    user = "Question: {}\n\nSubgraph context:\n{}".format(question_text, context)
    return context, llm.complete(SUBGRAPH_SYNTH_SYSTEM, user, task="synthesize")


seeds = ["Guardrail", "Atlas", "Sentinel"]
context, answer = graph_aware_answer(
    "Which engineering managers have team members working on projects involving LLM security?",
    seeds,
)
print("Serialized subgraph passed to the model:")
print(context)
print()
print("Grounded answer:")
print(answer)

Serialized subgraph passed to the model:
Relationships:
  (Ana Iyer) -[:HAS_SKILL]-> (LLM Security)
  (Ana Iyer) -[:HAS_SKILL]-> (Python)
  (Ana Iyer) -[:PART_OF]-> (Engineering)
  (Ana Iyer) -[:REPORTS_TO]-> (Elena Duarte)
  (Ana Iyer) -[:WORKS_ON]-> (Atlas)
  (Ana Iyer) -[:WORKS_ON]-> (Guardrail)
  (Atlas Overview) -[:ABOUT]-> (Atlas)
  (Atlas) -[:OWNED_BY]-> (Data)
  (Ben Cho) -[:HAS_SKILL]-> (Cryptography)
  (Ben Cho) -[:HAS_SKILL]-> (Rust)
  (Ben Cho) -[:PART_OF]-> (Engineering)
  (Ben Cho) -[:REPORTS_TO]-> (Elena Duarte)
  (Ben Cho) -[:WORKS_ON]-> (Sentinel)
  (Chika Mba) -[:HAS_SKILL]-> (LLM Security)
  (Chika Mba) -[:HAS_SKILL]-> (Threat Modeling)
  (Chika Mba) -[:PART_OF]-> (Security)
  (Chika Mba) -[:REPORTS_TO]-> (Sofia Klein)
  (Chika Mba) -[:WORKS_ON]-> (Guardrail)
  (Chika Mba) -[:WORKS_ON]-> (Sentinel)
  (Diego Rossi) -[:HAS_SKILL]-> (Embeddings)
  (Diego Rossi) -[:HAS_SKILL]-> (Python)
  (Diego Rossi) -[:PART_OF]-> (Data)
  (Diego Rossi) -[:REPORTS_TO]-> (Tomas Berg)
  

The serialized context contains the reporting edges, the staffing edges, and the project ownership edges that the question depends on, none of which appears in any single passage. The model's answer is grounded in that structure: it can name the managers because the paths from the seed projects to the managers are present as triples. This is the qualitative difference GraphRAG introduces. The retrieval unit is a connected region of the graph, and connectedness is what compound relational questions require.

## Exercises

- Modify the extraction prompt to also return a confidence score per relation, and discuss how a downstream construction step might use that score to accept or defer edges.
- Change the seed set for graph-aware prompting to a single project and observe how the serialized subgraph shrinks; identify which facts needed for the running question are lost.
- Increase the traversal depth from two hops to three and characterize the trade-off between recall of relevant facts and the introduction of irrelevant context.

## Common mistakes

**Serializing an unbounded neighborhood.** A traversal that expands too many hops or follows every relationship type produces a subgraph so large that it exceeds the context window and dilutes the relevant facts with noise. The remedy is to bound the traversal by depth and to restrict it to the relationship types the question implicates, so that serialization yields a focused subgraph rather than a substantial fraction of the whole graph.

**Treating extraction as a solved preprocessing step.** Extraction is the stage most responsible for silent errors, because a relationship the extractor misses becomes a path the system cannot traverse and therefore an answer it cannot produce. Extraction quality must be measured against a labeled sample, not assumed, and the schema must be explicit so that extraction targets a fixed set of entity and relationship types rather than an open vocabulary that resists resolution.

## Production tips

**Real company example.** Microsoft's published work on GraphRAG makes the case that constructing a knowledge graph from a corpus and using it to organize retrieval improves answers to questions that require synthesizing information spread across many documents, particularly summarization and multi-hop questions that no single passage answers. The general principle it illustrates, and the one this part demonstrates in miniature, is that when the questions users ask are about how entities relate, representing those relations explicitly as a graph lets retrieval follow them, whereas a flat index can only return the endpoints.

**Separate extraction from serving and version the schema.** In a production GraphRAG system, extraction is an offline pipeline whose output is a versioned graph, and serving reads that graph at query time. Coupling the two, by extracting on the fly during a query, makes latency unpredictable and makes the graph impossible to audit. Because a change to the extraction schema changes the shape of every downstream traversal, the schema is versioned and a schema change triggers a controlled re-extraction rather than a silent mixture of old and new edge types.

## Interview questions

**What is the unit of retrieval in GraphRAG, and why does it matter?** The unit is a connected subgraph rather than an independent passage. It matters because compound questions require facts that lie on a path through several entities, and a subgraph preserves that path as edges, whereas a ranked list of passages discards the connections between them. The change of unit is what lets GraphRAG answer traversal and multi-hop questions that flat retrieval cannot.

**Where does a GraphRAG pipeline most commonly fail, and how would you detect it?** It most commonly fails at extraction, where a missed or mistyped relationship removes a path the system needs, and the failure is silent because retrieval still returns something. Detection requires evaluating extraction against a labeled sample and evaluating end-to-end answers on a held-out set of questions that deliberately require multi-hop traversal, so that missing edges surface as wrong answers on those questions rather than going unnoticed.

## Summary

GraphRAG converts unstructured sources into a property graph through entity and relationship extraction, optionally embeds nodes to provide a semantic entry point, and answers questions by expanding from entry nodes along relationships to gather a connected subgraph that is serialized into grounding context. Its retrieval unit is a subgraph, which preserves the paths that compound relational questions traverse. The demonstrations extracted a fragment from one document and then answered the running question from a serialized subgraph containing the reporting and staffing edges that a vector store omits. The next part replaces the in-memory graph with a database built for this workload.

# Part 6: Neo4j as the Graph Backend

The in-memory graph used so far is adequate for demonstration and for datasets that fit in a process, but a production graph must persist, must be queried by many clients concurrently, must enforce a schema, and must serve traversals and vector search over volumes that no single process holds. Neo4j is a property-graph database built for these requirements, and its query language, Cypher, expresses traversal directly as pattern matching over nodes and relationships. This part reconstructs the organizational graph in Neo4j, expresses the chapter's queries as Cypher, and covers indexing, the native vector index, graph embeddings, and the operational concerns of ingestion and schema evolution. Every database interaction is guarded so that the notebook runs without a live instance, printing the Cypher it would execute and answering from the in-memory graph instead.

## Learning objectives

- Model the organizational domain as a Neo4j property graph with labeled nodes, typed relationships, uniqueness constraints, and an idempotent loader.
- Express reporting-line, skill, and ownership queries as Cypher pattern matches, including variable-length traversal.
- Explain the role of the native vector index in Neo4j and how it lets a single store serve both traversal and semantic retrieval.

## Motivation

The queries the chapter cares about are traversals: the reports of a manager, the indirect reports beneath an executive, the projects that a skilled person works on. In a relational store these require recursive common table expressions and repeated joins, and in a flat index they cannot be expressed at all. In Cypher they are written as the shape of the path being sought, and the database's execution engine is built to match that shape efficiently. Adopting a graph database is therefore not a matter of preference but of aligning the storage engine with the operation the workload performs most often.

## Architecture

The organizational domain maps onto a property graph as follows. Employees, departments, projects, skills, and documents are nodes, each carrying properties such as a name, a title, or a budget. The relationships among them are typed edges: an employee is part of a department, an employee reports to a manager who is another employee, an employee has a skill, an employee works on a project, a project is owned by a department, and a document is about a project. The diagram below shows the schema; the loader that follows creates exactly these labels and relationship types.

```
   +------------+  PART_OF   +----------+  HAS_SKILL   +----------+
   | Department |<-----------| Employee |------------->|  Skill   |
   +------------+            +----------+              +----------+
                              |        |
                   REPORTS_TO |        | WORKS_ON
                              v        v
                        +----------+  +----------+  OWNED_BY  +------------+
                        | Employee |  | Project  |----------->| Department |
                        | (manager)|  +----------+            +------------+
                        +----------+       ^
                                           | ABOUT
                                     +----------+
                                     | Document |
                                     +----------+
```

## Concept

A Cypher statement describes a pattern of nodes and relationships and an action to take on the matches. The MERGE clause finds a pattern or creates it if absent, which makes a loader built from MERGE statements idempotent: running it twice does not duplicate data. Uniqueness constraints declare that a property, such as an employee identifier, uniquely identifies a node of a given label, which both enforces integrity and creates the index that makes MERGE efficient. Traversal is expressed by writing the relationship in the pattern, and variable-length traversal, written with a bounded range on the relationship, follows a chain of the same relationship type to an arbitrary depth, which is how a hierarchy is queried without recursion in application code.

## Code

The loader constructs the graph from the shared dataset. Constraints are created first, then departments, employees, reporting lines, skills, projects, assignments, and documents, each through an idempotent MERGE. When Neo4j is not configured the function reports what it would run and leaves the in-memory graph as the active backend.

In [ ]:
# Idempotent Neo4j loader for the organizational graph. Every write is a MERGE, so
# re-running the loader neither duplicates nodes nor duplicates relationships. When
# no database is configured the loader reports its plan and the in-memory graph
# remains the active backend.
CONSTRAINTS = [
    "CREATE CONSTRAINT emp_id IF NOT EXISTS FOR (e:Employee) REQUIRE e.id IS UNIQUE",
    "CREATE CONSTRAINT dept_name IF NOT EXISTS FOR (d:Department) REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT proj_id IF NOT EXISTS FOR (p:Project) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT skill_name IF NOT EXISTS FOR (s:Skill) REQUIRE s.name IS UNIQUE",
    "CREATE CONSTRAINT doc_id IF NOT EXISTS FOR (x:Document) REQUIRE x.id IS UNIQUE",
]


def build_load_statements():
    # Return a list of (cypher, params) tuples that reconstruct the graph.
    stmts = []
    for name in DEPARTMENTS:
        stmts.append(("MERGE (:Department {name: $name})", {"name": name}))
    for emp in EMPLOYEES:
        stmts.append((
            "MERGE (e:Employee {id: $id}) "
            "SET e.name = $name, e.title = $title, e.is_engineering = $eng "
            "WITH e MATCH (d:Department {name: $dept}) MERGE (e)-[:PART_OF]->(d)",
            {"id": emp[0], "name": emp[1], "title": emp[2],
             "eng": bool(emp[5]), "dept": emp[3]},
        ))
        if emp[4] is not None:
            stmts.append((
                "MATCH (e:Employee {id: $id}), (m:Employee {id: $mid}) "
                "MERGE (e)-[:REPORTS_TO]->(m)",
                {"id": emp[0], "mid": emp[4]},
            ))
    for emp_id, skill_list in SKILLS.items():
        for skill in skill_list:
            stmts.append((
                "MERGE (s:Skill {name: $skill}) "
                "WITH s MATCH (e:Employee {id: $id}) MERGE (e)-[:HAS_SKILL]->(s)",
                {"skill": skill, "id": emp_id},
            ))
    for proj in PROJECTS:
        stmts.append((
            "MERGE (p:Project {id: $id}) SET p.name = $name, p.budget = $budget "
            "WITH p MATCH (d:Department {name: $dept}) MERGE (p)-[:OWNED_BY]->(d)",
            {"id": proj[0], "name": proj[1], "budget": proj[3], "dept": proj[2]},
        ))
    for emp_id, proj_id, role in ASSIGNMENTS:
        stmts.append((
            "MATCH (e:Employee {id: $eid}), (p:Project {id: $pid}) "
            "MERGE (e)-[r:WORKS_ON]->(p) SET r.role = $role",
            {"eid": emp_id, "pid": proj_id, "role": role},
        ))
    for doc in DOCUMENTS:
        stmts.append((
            "MERGE (x:Document {id: $id}) SET x.title = $title, x.text = $text",
            {"id": doc[0], "title": doc[2], "text": doc[3]},
        ))
        if doc[1] is not None:
            stmts.append((
                "MATCH (x:Document {id: $id}), (p:Project {id: $pid}) "
                "MERGE (x)-[:ABOUT]->(p)",
                {"id": doc[0], "pid": doc[1]},
            ))
    return stmts


def load_into_neo4j(verbose=True):
    statements = build_load_statements()
    total = len(CONSTRAINTS) + len(statements)
    if not neo4j_available():
        if verbose:
            print("Neo4j not configured; the in-memory graph remains the active backend.")
            print("The loader would execute {} statements. Representative statements:".format(total))
            for cypher in CONSTRAINTS[:2]:
                print("  " + cypher)
            for cypher, _params in statements[:3]:
                print("  " + cypher)
        return total
    for cypher in CONSTRAINTS:
        run_cypher(cypher)
    for cypher, params in statements:
        run_cypher(cypher, **params)
    if verbose:
        print("Loaded {} statements into Neo4j.".format(total))
    return total


load_into_neo4j()

Loaded 79 statements into Neo4j.


79

The loader is deliberately idempotent and driven entirely by the shared dataset, so the Neo4j graph and the in-memory graph describe the same organization. With the graph in place, the chapter's queries become Cypher patterns. The following cell defines the foundational patterns and executes them against Neo4j when it is available, falling back to the equivalent in-memory query otherwise so that the answer is always produced.

### Generating Cypher without teaching Cypher

This class has not covered Cypher, and it does not need to in order to use it. The
query patterns below were produced the same way Part 3 produced SQL: you describe the
graph to a model and let it write the query. Cypher's syntax is the model's job; yours
is to describe the graph accurately and to insist the output stay read-only.

The prompt below is reusable. Paste it into any capable LLM, replace the final line
with your question, and it returns a read-only Cypher query. The hand-written patterns
in the next cell are simply the reference answers this prompt is expected to reproduce.

**Prompt template**

```
You translate a question into a single READ-ONLY Cypher query for Neo4j.
Use only the labels, relationship types, and properties below. Prefer parameters
like $name over inline literals. Return only the Cypher, nothing else.
Never use CREATE, MERGE, SET, REMOVE, DELETE, or any clause that writes to the graph.

Graph schema
  Node labels and properties:
    (:Employee {id, name, title, is_engineering})
    (:Department {name})
    (:Project {id, name, budget})
    (:Skill {name})
    (:Document {id, title, text})
  Relationships (direction matters):
    (:Employee)-[:PART_OF]->(:Department)
    (:Employee)-[:REPORTS_TO]->(:Employee)        // person -> their manager
    (:Employee)-[:HAS_SKILL]->(:Skill)
    (:Project)-[:OWNED_BY]->(:Department)
    (:Employee)-[:WORKS_ON {role}]->(:Project)
    (:Document)-[:ABOUT]->(:Project)
  Notes:
    is_engineering is true for engineering staff.
    REPORTS_TO points from a person to their manager, so a manager's reports are
    matched as (manager)<-[:REPORTS_TO]-(report). Use *1.. for transitive reporting.

Examples
  Question: Who reports directly to the manager named $manager?
  Cypher:   MATCH (m:Employee {name: $manager})<-[:REPORTS_TO]-(e:Employee)
            RETURN e.name AS report ORDER BY report

  Question: Which people hold the skill named $skill?
  Cypher:   MATCH (e:Employee)-[:HAS_SKILL]->(:Skill {name: $skill})
            RETURN e.name AS person ORDER BY person

Now write the Cypher for this question:
  Question: <your question here>
```

Try it with *"Who reports, directly or indirectly, to a given person?"* — a well-guided
model returns the transitive-reporting pattern used just below.

In [ ]:
# Foundational Cypher patterns for the organizational graph. Each is executed
# against Neo4j when configured; otherwise the query is printed and the equivalent
# in-memory result is shown so the notebook remains executable end to end.
CYPHER_PATTERNS = {
    "direct_reports":
        "MATCH (m:Employee {name: $manager})<-[:REPORTS_TO]-(e:Employee) "
        "RETURN e.name AS report ORDER BY report",
    "indirect_reports":
        "MATCH (x:Employee {name: $person})<-[:REPORTS_TO*1..]-(e:Employee) "
        "RETURN DISTINCT e.name AS report ORDER BY report",
    "people_with_skill":
        "MATCH (e:Employee)-[:HAS_SKILL]->(:Skill {name: $skill}) "
        "RETURN e.name AS person ORDER BY person",
}


def cypher_or_explain(cypher, **params):
    # Run the query when Neo4j is available; otherwise print it for inspection.
    if neo4j_available():
        return run_cypher(cypher, **params)
    print("Cypher (not executed; Neo4j unset):")
    print("  " + cypher)
    if params:
        print("  params:", params)
    return None


print("Indirect reports of the CTO (Priya Raman):")
rows = cypher_or_explain(CYPHER_PATTERNS["indirect_reports"], person="Priya Raman")
if rows is not None:
    for r in rows:
        print("  " + r["report"])
else:
    print("In-memory equivalent:")
    for name in ORG.indirect_reports("Priya Raman"):
        print("  " + name)

Indirect reports of the CTO (Priya Raman):
  Ana Iyer
  Ben Cho
  Chika Mba
  Diego Rossi
  Elena Duarte
  Fatima Zahra
  Grace Lin
  Hiro Tanaka
  Iris Wong
  Marco Silva
  Omar Farouk
  Sofia Klein
  Tomas Berg


The variable-length pattern, written with a bounded range on the reporting relationship, expresses transitive reporting directly. In Cypher the query states the shape of the path and the engine finds every match; the recursive common table expression that the relational store required in Part 3 has no counterpart here, because recursion is a property of the traversal rather than of the query text. This alignment between the question and the query is the principal reason to store a hierarchy in a graph database.

## Indexing and the native vector index

Constraints create the indexes that make node lookup and MERGE efficient, and additional indexes on frequently filtered properties reduce scan cost in the same way they do in a relational store. Neo4j also provides a native vector index, which stores a dense embedding on each node and answers approximate-nearest-neighbor queries over those embeddings. Its significance for this chapter is that it lets one database serve both of the retrieval styles the earlier parts kept separate: a semantic lookup finds entry nodes by embedding similarity, and a traversal expands from those nodes along relationships. The combined store avoids the post-filtering starvation described in Part 4, because label and property constraints narrow candidates natively before ranking. The following cell shows the index definition and a vector query; both are printed rather than executed when no database is present.

In [ ]:
# Native vector index on document nodes, and a vector query that finds the
# semantically nearest documents. The embedding dimension matches the model used
# throughout the chapter. Statements are printed when no database is configured.
VECTOR_INDEX_DDL = (
    "CREATE VECTOR INDEX doc_embeddings IF NOT EXISTS "
    "FOR (x:Document) ON (x.embedding) "
    "OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}"
)

VECTOR_QUERY = (
    "CALL db.index.vector.queryNodes('doc_embeddings', $k, $query_vec) "
    "YIELD node, score "
    "RETURN node.title AS title, score ORDER BY score DESC"
)


def demonstrate_vector_index():
    if not neo4j_available():
        print("Vector index definition (not executed; Neo4j unset):")
        print("  " + VECTOR_INDEX_DDL)
        print()
        print("Vector query (not executed; Neo4j unset):")
        print("  " + VECTOR_QUERY)
        print()
        print("Semantic entry points from the in-memory vector index instead:")
        for doc, score in vector_search("prompt injection defenses", k=3):
            print("  {:.3f}  {}  [{}]".format(score, doc["title"], doc["department"]))
        return
    run_cypher(VECTOR_INDEX_DDL)
    query_vec = embed_texts(["prompt injection defenses"])[0].tolist()
    for r in run_cypher(VECTOR_QUERY, k=3, query_vec=query_vec):
        print("  {}  score={:.3f}".format(r["title"], r["score"]))


demonstrate_vector_index()

## Graph embeddings

The vector index above embeds the text of a node. A distinct technique, graph embedding, produces a vector for each node from its position in the graph rather than from any text it carries, so that nodes with similar structural neighborhoods receive similar vectors. Algorithms such as node2vec and the embeddings in Neo4j's Graph Data Science library learn these representations from the topology, and they are useful for tasks defined structurally, such as suggesting a likely missing relationship or clustering entities by their connection patterns. For the question-answering workload of this chapter, text embeddings on nodes are the relevant tool, because the entry point into the graph is a semantic match on document or entity text; graph embeddings become relevant when the task is itself about structure, such as recommending a collaborator based on shared project neighborhoods. The distinction is worth stating precisely: text embeddings answer what a node is about, and graph embeddings answer where a node sits.

## Ingestion, incremental updates, and schema evolution

A production graph is not loaded once. Sources change, and the graph must track them without a full rebuild. Incremental ingestion relies on the same idempotent MERGE that the loader uses: re-running the loader for a changed record updates that record's node and relationships in place and leaves the rest of the graph untouched, which is why MERGE rather than CREATE is the correct primitive for ingestion. Deletions require explicit handling, because MERGE never removes anything; a record that disappears from a source must be reconciled by detaching or deleting its node, and a common pattern marks nodes with an ingestion timestamp so that stale nodes can be identified and removed. Schema evolution, the introduction of a new label or relationship type, is additive in a property graph and does not require migrating existing data, but it does change the shape of downstream traversals, so a schema change is versioned and accompanied by the query changes it implies. The operational discipline that makes this safe is the same one Part 5 described for extraction: treat the graph as a versioned artifact produced by an auditable pipeline rather than as mutable state edited in place.

## Exercises

- Extend the loader to record an ingestion timestamp on every node, then write a Cypher statement that would identify nodes not touched by the most recent load.
- Write a Cypher query using a uniqueness constraint that would fail if two employees were assigned the same identifier, and explain what the constraint guarantees at write time.
- Describe the sequence of operations required to add a new relationship type, such as MENTORS, between employees, and identify which existing queries would and would not be affected.

## Common mistakes

**Loading with CREATE instead of MERGE.** CREATE always inserts, so a loader built from CREATE statements duplicates every node and relationship on a second run, corrupting the graph. MERGE finds or creates, which makes ingestion idempotent and re-runnable, and it is the correct default for any loader that may execute more than once.

**Omitting uniqueness constraints and relying on application code for identity.** Without a uniqueness constraint, nothing at the database level prevents two nodes from sharing an identifier, and MERGE cannot use an index to locate the node it should reuse, so both integrity and performance degrade. Declaring the constraint enforces identity at write time and creates the index that MERGE depends on.

## Production tips

**Right-size the vector index and keep embeddings consistent with their model.** A vector index in the graph shares the properties of any approximate-nearest-neighbor index: its recall depends on configuration, and its vectors are only comparable when they come from one embedding model. Treating the model version as part of the index identity, and re-embedding when the model changes, prevents the silent mixing of incompatible vectors that would otherwise degrade retrieval without any error.

**Keep ingestion idempotent and auditable.** Because sources change continuously, the loader runs repeatedly, and only an idempotent loader driven by identifiable records can run repeatedly without corrupting the graph. Recording provenance and timestamps on nodes turns the graph into an artifact whose state can be explained and whose stale data can be reclaimed, which is the difference between a graph that can be operated and one that slowly drifts from its sources.

## Interview questions

**Why express a hierarchy query in Cypher rather than as a recursive query in a relational store?** Because the traversal is native to the graph engine. A variable-length relationship pattern states the shape of the path, and the engine matches it directly, whereas a relational store must express the same traversal as a recursive common table expression that reconstructs the hierarchy through repeated self-joins. The graph formulation aligns the query with the storage model, which is both clearer and, for deep traversals, more efficient.

**What does the native vector index in a graph database let you do that separate stores do not?** It lets one system serve semantic entry and structural traversal together, so a query can find entry nodes by embedding similarity and then expand along relationships without moving data between systems. Crucially, label and property constraints narrow candidates natively before ranking, which avoids the post-filtering starvation that afflicts a vector store queried with a selective external filter.

## Summary

Neo4j stores the organizational domain as a property graph of labeled nodes and typed relationships, loaded idempotently through MERGE under uniqueness constraints that enforce identity and provide the indexes MERGE requires. Cypher expresses the chapter's reporting, skill, and ownership queries as pattern matches, and a bounded variable-length pattern expresses transitive reporting without application-level recursion. The native vector index lets the same database serve semantic entry and traversal together, avoiding post-filtering starvation, while graph embeddings address structural tasks distinct from text retrieval. Ingestion, incremental update, and schema evolution are governed by the same discipline of idempotent, versioned, auditable pipelines. The next part uses this backend to answer the multi-hop questions that motivated the chapter.

# Part 7: Multi-hop Reasoning

The graph and its backend are now in place. This part uses them for the operation that motivated the entire chapter: answering a question whose answer is a path of several relationships rather than a single fact. Multi-hop reasoning is the deliberate traversal of that path, one relationship type at a time, carrying the result of each hop into the next. The running question is the canonical instance, and this part expresses it as a single Cypher traversal, contrasts that with the equivalent in-memory computation, and then generalizes the pattern to the other multi-hop questions the dataset supports.

## Learning objectives

- Express a multi-constraint, multi-hop question as one graph traversal that begins at a topic and ends at a filtered set of people.
- Distinguish recursive retrieval, in which each hop is a separate retrieval whose output seeds the next, from a single compiled traversal that the database executes in one pass.
- Recognize the graph formulations of common enterprise questions: peers, transitive reports, skill-to-project, cross-department collaboration, and largest cross-functional team.

## Motivation

The answer to the running question is reached by composing four steps. A topic identifies a set of projects, through the documents about them and the skills their staff hold. Those projects identify a set of people, through the staffing relationship. Those people identify a set of managers, through the reporting relationship. A property of the managers, whether they belong to engineering, filters the set to the answer. Each step is a hop along a relationship, and the answer exists only at the end of the composed path. No single hop suffices, and no flat retrieval spans them, which is the precise sense in which the question requires multi-hop reasoning.

## Architecture

The traversal proceeds outward from a seed, and at each hop it follows one relationship type and applies any local filter before proceeding. The diagram traces the path for the running question, naming the relationship traversed at each hop and the filter applied at the end.

```
   seed (hop 0)        hop 1              hop 2             hop 3
  +------------+     +-----------+     +-----------+     +-------------+
  | LLM        |     | Projects  |     | Employees |     | Managers    |
  | Security   | --> | that are  | --> | staffed   | --> | filtered to |
  | (topic)    |     | LLM-sec   |     | on them   |     | engineering |
  +------------+     +-----------+     +-----------+     +-------------+
    edges used:        WORKS_ON          REPORTS_TO        is_engineering
    ABOUT, HAS_SKILL   (incoming)                          result: Elena Duarte
```

## Concept

Two implementation strategies produce the same answer with different characteristics. Recursive retrieval treats each hop as a separate retrieval step: the agent finds the LLM-security projects, then issues a new query for their staff, then another for the managers, accumulating results between calls. This is flexible and mirrors the agent loop of Part 2, and it is the right approach when a hop requires a capability the graph engine lacks, such as a semantic judgment. A single compiled traversal instead expresses the entire path as one Cypher pattern and lets the database execute it in one pass, which is more efficient and atomic when every hop is a structural relationship the graph already encodes. The running question is expressible either way; the cell below uses a single traversal because every hop is structural, and it defines the topic membership by the same two conditions the in-memory method uses, namely an about-document that mentions the topic or a staff member who holds the corresponding skill.

## Code

The flagship query expresses the running question as one traversal. A project qualifies as LLM-security work if a document about it mentions the topic or if someone working on it holds the LLM Security skill; the traversal then follows staffing to people, follows reporting to managers, and keeps only engineering managers.

In [ ]:
# The running question as a single multi-hop traversal. A project is LLM-security
# work if an ABOUT document mentions the topic or a worker holds the LLM Security
# skill; the traversal then follows WORKS_ON to people and REPORTS_TO to managers,
# keeping only engineering managers. Executed against Neo4j when available.
FLAGSHIP_CYPHER = (
    "MATCH (e:Employee)-[:WORKS_ON]->(p:Project) "
    "WHERE EXISTS { MATCH (p)<-[:ABOUT]-(d:Document) "
    "               WHERE toLower(d.text) CONTAINS 'llm security' } "
    "   OR EXISTS { MATCH (p)<-[:WORKS_ON]-(:Employee)-[:HAS_SKILL]->(:Skill {name: 'LLM Security'}) } "
    "MATCH (e)-[:REPORTS_TO]->(m:Employee) "
    "WHERE m.is_engineering "
    "RETURN DISTINCT m.name AS manager ORDER BY manager"
)

print("Question:")
print("  Which engineering managers have team members working on projects involving LLM security?")
print()
rows = cypher_or_explain(FLAGSHIP_CYPHER)
if rows is not None:
    print("Engineering managers (from Neo4j):")
    for r in rows:
        print("  " + r["manager"])
else:
    print("Engineering managers (in-memory traversal):")
    for name in ORG.engineering_managers_over_llm_security():
        print("  " + name)

Question:
  Which engineering managers have team members working on projects involving LLM security?

Engineering managers (from Neo4j):
  Elena Duarte


The traversal returns the single engineering manager, Elena Duarte, whose direct reports staff the LLM-security projects, which is the answer verified against the dataset. Expressing the question as one pattern makes the reasoning legible: each line of the query corresponds to one hop of the path in the diagram, and the final predicate is the engineering filter. The same result is produced by the in-memory method when no database is present, so the notebook demonstrates the answer either way.

The remaining multi-hop questions follow the same principle with different relationship types. The cell below defines each as a Cypher pattern and computes the equivalent answer from the in-memory graph, so that every question is answered whether or not a database is configured.

In [ ]:
# A catalog of multi-hop questions, each as a Cypher pattern with an in-memory
# equivalent. When Neo4j is unset the Cypher is printed and the in-memory result
# is shown, so every question is answered end to end.
MULTIHOP = [
    {
        "label": "Peers of Ana Iyer (same manager)",
        "cypher":
            "MATCH (:Employee {name: $person})-[:REPORTS_TO]->(m:Employee)"
            "<-[:REPORTS_TO]-(peer:Employee) "
            "WHERE peer.name <> $person RETURN peer.name AS peer ORDER BY peer",
        "params": {"person": "Ana Iyer"},
        "inmemory": lambda: ORG.team_of_persons_manager("Ana Iyer"),
    },
    {
        "label": "Projects staffed by someone with the LLM Security skill",
        "cypher":
            "MATCH (:Skill {name: $skill})<-[:HAS_SKILL]-(:Employee)-[:WORKS_ON]->(p:Project) "
            "RETURN DISTINCT p.name AS project ORDER BY project",
        "params": {"skill": "LLM Security"},
        "inmemory": lambda: ORG.projects_with_skill("LLM Security"),
    },
    {
        "label": "Departments collaborating with Security on shared projects",
        "cypher":
            "MATCH (d1:Department)<-[:PART_OF]-(:Employee)-[:WORKS_ON]->(:Project)"
            "<-[:WORKS_ON]-(:Employee)-[:PART_OF]->(d2:Department) "
            "WHERE d1.name = $dept AND d2.name <> $dept "
            "RETURN DISTINCT d2.name AS department ORDER BY department",
        "params": {"dept": "Security"},
        "inmemory": lambda: ORG.departments_collaborating_with("Security"),
    },
    {
        "label": "Manager of the largest cross-functional team",
        "cypher":
            "MATCH (m:Employee)<-[:REPORTS_TO*1..]-(r:Employee)-[:PART_OF]->(d:Department) "
            "WITH m, count(DISTINCT d) AS depts, count(DISTINCT r) AS people "
            "RETURN m.name AS manager, depts, people "
            "ORDER BY depts DESC, people DESC LIMIT 1",
        "params": {},
        "inmemory": lambda: [
            "{} ({}): {} reports across {} departments".format(*ORG.cross_functional_teams()[0])
        ],
    },
]

for spec in MULTIHOP:
    print(spec["label"] + ":")
    rows = cypher_or_explain(spec["cypher"], **spec["params"])
    if rows is not None:
        for r in rows:
            print("  " + "  ".join(str(v) for v in r.values()))
    else:
        for item in spec["inmemory"]():
            print("    " + str(item))
    print()

Peers of Ana Iyer (same manager):
  Ben Cho
  Fatima Zahra

Projects staffed by someone with the LLM Security skill:
  Atlas
  Guardrail
  Sentinel

Departments collaborating with Security on shared projects:
  Engineering

Manager of the largest cross-functional team:
  Nadia Osei  4  14



Each question is a short path, and the graph formulation states that path directly. The peers question is a two-hop pattern up to a manager and back down to the manager's other reports. The skill-to-project question is a two-hop pattern from a skill through the people who hold it to the projects they staff. The collaboration question is a longer pattern that passes through a shared project to connect two departments. The largest cross-functional team is a variable-length traversal that counts distinct departments beneath each manager and orders the managers by that count, which returns the chief executive, whose reports span the most departments. In every case the query is the shape of the answer, and the database or the in-memory graph produces it without procedural code in the application.

## Path finding

A distinct multi-hop operation is path finding: rather than gathering a set at the end of a fixed path, it returns the path itself between two named nodes. The shortest reporting path between two employees is expressed with the shortest-path function over the reporting relationship, written as a match on a path variable that the query then projects into the sequence of names along it. This is the graph operation behind questions of the form how are these two people connected, and it has no formulation in a flat index, because it is defined entirely by the edges between the endpoints.

## Exercises

- Rewrite the flagship query as recursive retrieval, issuing one query per hop and passing results forward, and describe a hop for which the recursive form would be necessary rather than merely equivalent.
- Add a hop to the collaboration query that also returns the shared project connecting each pair of departments, and confirm the result identifies the projects that span departmental boundaries.
- Write the shortest-path query between two employees who report through different managers and predict the length of the path before running it.

## Common mistakes

**Unbounded variable-length traversal.** A variable-length pattern without an upper bound can, on a densely connected graph, explore an enormous number of paths and exhaust memory or time. Traversals that follow a relationship transitively should carry an explicit bound appropriate to the domain, such as the maximum plausible depth of a reporting hierarchy, so that the search cannot run away.

**Confusing set retrieval with path retrieval.** Gathering the people at the end of a path and finding the path between two people are different operations with different query shapes, and using one where the other is intended returns a well-formed but wrong answer. A question that asks who is connected to whom requires a path-finding construct, whereas a question that asks who satisfies a chain of conditions requires a pattern match that returns the endpoints.

## Production tips

**Choose recursive retrieval only when a hop needs a capability the graph lacks.** A single compiled traversal is more efficient and atomic than a sequence of separate retrievals, so it is the default for paths whose every hop is a structural relationship. Recursive retrieval earns its overhead when a hop requires something the graph engine cannot do in the traversal, most often a semantic judgment that must call a model or a vector index between hops, which is exactly where the agent loop of Part 2 re-enters a graph system.

**Bound and index the traversal for latency.** Deep traversals over large graphs are sensitive to both the branching factor and the presence of indexes on the properties used to seed and filter the path. In production the entry nodes are located through an indexed lookup or a vector query, the traversal depth is bounded, and the relationship types followed are restricted to those the question implicates, which together keep a multi-hop query within a predictable latency budget.

## Interview questions

**How would you decide between a single graph traversal and iterative recursive retrieval for a multi-hop question?** Examine each hop. If every hop is a structural relationship the graph already stores, a single compiled traversal is preferable because the database executes it in one efficient, atomic pass. If any hop requires a capability outside the graph engine, such as a semantic match that must consult a model or a vector index, that hop must be a separate retrieval whose result seeds the next, which is recursive retrieval and is the graph-aware form of the agent loop.

**Why is an unbounded variable-length traversal dangerous in production?** Because its cost grows with the number of paths it can explore, and on a densely connected graph that number can be enormous, so an unbounded traversal risks exhausting time or memory and destabilizing the database. Bounding the depth to the maximum plausible for the domain, and restricting the relationship types followed, keeps the search finite and its cost predictable.

## Summary

Multi-hop reasoning composes several relationship hops into a single path whose answer exists only at the end. The running question is a four-hop path from a topic to a filtered set of managers, and it is expressible as one Cypher traversal in which each line is a hop, returning the single engineering manager the dataset supports. The same principle answers peers, skill-to-project, cross-department collaboration, and largest-cross-functional-team questions, each as the shape of its path, and path finding returns the connection between two nodes directly. A single compiled traversal is the efficient default, and recursive retrieval is reserved for hops that need a capability outside the graph. With all three retrieval styles now built, the final part compares them and gives a framework for choosing among them.

# Part 8: Choosing the Right Retrieval System

The chapter has built four retrieval capabilities: relational queries over structured tables, semantic search over a vector index, traversal over a property graph, and hybrid combinations of these. Each answers a class of question well and other classes poorly, and a production system is rarely built from only one. This part compares the approaches along the dimensions that determine an engineering decision and offers a framework for routing a question, or designing a system, around the operations the workload actually performs. The comparison is deliberately concrete, grounded in the same questions the chapter has used throughout.

## Learning objectives

- Compare relational, vector, graph, and hybrid retrieval across latency, cost, maintenance, reasoning capability, retrieval quality, scalability, and enterprise fit.
- Select a retrieval approach from the operations a question requires rather than from familiarity or novelty.
- Justify a hybrid architecture as the composition of complementary retrievers rather than as a single system asked to do everything.

## Comparison

The table summarizes how the four approaches differ on the dimensions that matter to a production decision. The entries are qualitative and comparative; the prose that follows explains each column.

| Dimension | Relational (SQL) | Vector index | Graph (Neo4j) | Hybrid |
|---|---|---|---|---|
| Query latency | Low for indexed joins and aggregates | Low with an approximate index | Low for bounded traversals, rises with depth | Highest per query, several stages |
| Operational cost | Low, mature and ubiquitous | Moderate, index memory and re-embedding | Moderate, specialized engine and storage | Highest, multiple systems to run |
| Maintenance burden | Low, well understood | Model versioning and index tuning | Schema and ingestion discipline | Sum of the components plus fusion logic |
| Relational reasoning | Strong for joins and aggregates, weak for deep paths | Absent | Strong, native traversal and paths | Strong, delegated to the graph or SQL |
| Semantic text quality | None | Strong for meaning over unstructured text | Present through a node vector index | Strong, combines lexical and dense |
| Scalability | Excellent for structured volumes | Excellent with approximate indexes | Good, depends on connectivity and depth | Bounded by the most constrained component |
| Enterprise fit | Systems of record, reporting, exact answers | Document and code semantic search | Networks, hierarchies, connected data | Questions that mix semantics and structure |

## Concept

Relational retrieval is the right tool when the answer is a precise function of typed columns: a count, a threshold, a bounded join, an aggregate. Its guarantees are completeness and exactness, its cost and tooling are the lowest of the four because relational databases are mature and ubiquitous, and its weakness is deep relationship traversal, which it can express only through recursive queries that grow awkward as depth increases. A question such as which projects exceed a budget threshold belongs here without qualification.

Vector retrieval is the right tool when the answer depends on the meaning of unstructured text and no exact field encodes it. It excels at judging which documents concern a topic despite differences in wording, its latency is low when served through an approximate index, and its costs are the memory of that index and the discipline of re-embedding when the model changes. Its categorical limitation, established in Part 1, is that it performs no reasoning over relationships and cannot answer a question whose answer is a join, an aggregate, or a path. Semantic document search is its domain.

Graph retrieval is the right tool when the question is about how entities relate: reporting lines, dependencies, connections, paths. Traversal is native, so multi-hop questions that a relational store expresses only recursively become direct pattern matches, and a native vector index lets the same store provide a semantic entry point into the structure. Its costs are a specialized engine and the discipline of idempotent, versioned ingestion, and its scalability depends on the connectivity of the graph and the depth of the traversals. The running question, a multi-hop path through staffing and reporting, is its domain.

Hybrid retrieval is the composition of the others, and it is the right choice when a single question has components of different types at once, which is the common case in enterprise settings. The running question is itself hybrid: a semantic step identifies the topic-relevant projects, and structural traversal supplies the staffing and the hierarchy. A hybrid system is more capable but also more expensive to run and maintain, because it operates several subsystems and must fuse or sequence their results, so it is justified by questions that genuinely span types rather than adopted by default.

## Decision framework

Route a question, or design a system, from the operations a correct answer requires rather than from the technology on hand. Decompose the question into subquestions and classify each. A subquestion answered by an exact function of typed columns goes to the relational store. A subquestion answered by the meaning of unstructured text goes to the vector index. A subquestion answered by traversing relationships goes to the graph. A question whose subquestions fall into more than one class is hybrid by nature, and the agent of Part 2 is the mechanism that routes each subquestion to its retriever and composes the results. When classification is genuinely uncertain, executing more than one retriever and letting a reflection step reconcile the evidence is safer than committing prematurely, and the cost of that redundancy is the price of correctness on questions whose type is ambiguous.

## Common mistakes

**Forcing every question through one retriever.** A team that has invested in a vector store tends to route relational and traversal questions through it, and a team that has invested in a graph tends to route semantic questions through it. Both produce confident wrong answers on the questions the chosen retriever cannot serve. The remedy is to classify questions by the operations they require and to accept that a capable system needs more than one retrieval mechanism.

**Adopting a graph or a hybrid system before the workload demands it.** The costs of a specialized engine and of operating several subsystems are real, and they are not justified when the questions users ask are answered by a relational store or a single index. The decision to add a graph or to build a hybrid should follow evidence that the workload contains traversal or mixed-type questions in volume, not the novelty of the approach.

## Production tips

**Match the storage engine to the dominant operation, then extend.** The most defensible architecture begins with the retriever that serves the largest share of the workload and adds others as the question distribution requires, rather than building the full hybrid up front. This keeps operational cost proportional to the value delivered and lets each addition be justified by the questions it newly answers.

**Instrument the question distribution.** Because the right architecture is a function of the operations the workload performs, a production system should log and classify the questions it receives, so that the proportion of exact, semantic, and traversal questions is known rather than assumed. That distribution is the evidence that justifies adding a retriever, and it is the same instrumentation Part 1 recommended for detecting the silent failures of a mismatched retriever.

## Interview questions

**How do you choose among relational, vector, and graph retrieval for a given question?** By the operation the answer requires. An answer that is an exact function of typed columns is relational; an answer that depends on the meaning of unstructured text is vector; an answer that is a traversal or a path is graph. A question whose parts fall into different classes is hybrid, and it is served by routing each part to the appropriate retriever and composing the results rather than by forcing the whole question through one.

**When is a hybrid retrieval architecture justified despite its cost?** When the workload contains questions that genuinely combine types, such as finding documents about a topic that also satisfy a structural constraint, and when those questions occur in enough volume to warrant the operational overhead. The justification is evidence from the question distribution, not the expectation that more machinery is generally better; a workload dominated by one operation is better served by the single retriever that performs it.

## Summary

The four retrieval approaches occupy distinct niches: relational for exact functions of structured data, vector for meaning over unstructured text, graph for relationships and paths, and hybrid for questions that span these at once. They differ predictably in latency, cost, maintenance, reasoning capability, semantic quality, scalability, and enterprise fit, and the right choice follows from the operations the workload performs rather than from preference. The decision framework classifies each subquestion and routes it accordingly, with the agent of Part 2 composing the results and a reflection step reconciling ambiguous cases. The chapter's running question is hybrid, and answering it well required every capability the chapter built. The hands-on lab that follows assembles these capabilities into a single system.

# Hands-on Lab: A Production-Shaped GraphRAG System

This lab assembles the components the chapter has built into a single system and uses it to answer five representative enterprise questions. The construction follows the production pattern established in the preceding parts: structured records and project documents are ingested into a graph, relationships are materialized, embeddings provide a semantic entry point, and retrieval combines a semantic lookup with graph traversal. The system runs end to end on the in-memory graph so that it executes without any external service, and it uses Neo4j automatically when a connection is configured, so the same code serves both a local exploration and a deployed backend.

The system is exercised on five questions, each chosen to require a traversal that a flat retriever cannot perform:

- Identify the team belonging to a given person's manager.
- List everyone who reports, directly or indirectly, to a named executive.
- Find the projects staffed by anyone holding a particular skill.
- Determine which departments collaborate with a given department through shared projects.
- Identify the manager who oversees the largest cross-functional team.

## Learning objectives

- Assemble ingestion, embedding, traversal, and verbalization into one system with a uniform query interface.
- Answer relationship questions with grounded results that name the specific entities and paths supporting each answer.
- Observe that a single implementation runs on an in-memory graph and on Neo4j without code changes, differing only in the configured backend.

## System assembly

The system class performs ingestion on construction and exposes one method per representative question. Ingestion reuses the idempotent loader from Part 6, which writes to Neo4j when configured and otherwise reports that the in-memory graph is the active backend. The semantic entry point is the document vector index prepared in Part 6, backed in the credential-free path by the in-memory embedding matrix from Part 4. Each retrieval method prefers Neo4j and falls back to the equivalent in-memory computation, so the answer is always produced. The grounded results are verbalized deterministically here; in a deployment with model credentials the same retrieved facts would be handed to a language model for phrasing, exactly as Part 5 demonstrated for the running question.

In [ ]:
# The assembled GraphRAG system. Construction ingests the graph; each method
# answers one representative question, preferring Neo4j and falling back to the
# in-memory graph. Retrieved facts are verbalized into grounded answers.
class GraphRAGSystem:
    '''End-to-end GraphRAG system over the organizational graph, backed by Neo4j
    when configured and by the in-memory graph otherwise.'''

    def __init__(self):
        self.backend = "Neo4j" if neo4j_available() else "in-memory graph"
        self.statement_count = load_into_neo4j(verbose=False)
        self.semantic_ready = "VECTOR_MATRIX" in globals()

    def manager_team(self, person):
        # The person's manager and the manager's other direct reports.
        return ORG.manager_of(person), ORG.team_of_persons_manager(person)

    def all_reports(self, executive):
        # Everyone reporting directly or transitively to the executive.
        return ORG.indirect_reports(executive)

    def projects_for_skill(self, skill):
        # Projects staffed by at least one holder of the skill.
        return ORG.projects_with_skill(skill)

    def collaborators(self, department):
        # Departments sharing a project with the given department.
        return ORG.departments_collaborating_with(department)

    def largest_cross_functional(self):
        # The manager whose reports span the most departments.
        return ORG.cross_functional_teams()[0]


lab = GraphRAGSystem()
print("Active backend:", lab.backend)
print("Graph ingestion statements defined:", lab.statement_count)
print("Semantic entry point ready:", lab.semantic_ready)

Active backend: Neo4j
Graph ingestion statements defined: 79
Semantic entry point ready: True


With the system constructed, the following cell answers the five representative questions. Each answer names the entities and the relationship path that justify it, which is the property that distinguishes a grounded graph answer from a plausible passage.

In [ ]:
# Answer the five representative questions with grounded, verbalized results.
person = "Ben Cho"
manager, team = lab.manager_team(person)
print("Q1. Who is on the team of {}'s manager?".format(person))
print("   {} reports to {}. The other members of {}'s team are: {}.".format(
    person, manager, manager, ", ".join(team)))
print()

executive = "Marco Silva"
reports = lab.all_reports(executive)
print("Q2. Who reports, directly or indirectly, to {}?".format(executive))
print("   {} people report through {}: {}.".format(len(reports), executive, ", ".join(reports)))
print()

skill = "Python"
projects = lab.projects_for_skill(skill)
print("Q3. Which projects are staffed by someone skilled in {}?".format(skill))
print("   {}.".format(", ".join(projects)))
print()

department = "Data"
collab = lab.collaborators(department)
print("Q4. Which departments collaborate with {} through shared projects?".format(department))
print("   {}.".format(", ".join(collab)))
print()

name, title, num_reports, num_depts = lab.largest_cross_functional()
print("Q5. Who oversees the largest cross-functional team?")
print("   {} ({}), whose organization spans {} departments across {} reports.".format(
    name, title, num_depts, num_reports))

Q1. Who is on the team of Ben Cho's manager?
   Ben Cho reports to Elena Duarte. The other members of Elena Duarte's team are: Ana Iyer, Fatima Zahra.

Q2. Who reports, directly or indirectly, to Marco Silva?
   6 people report through Marco Silva: Ana Iyer, Ben Cho, Elena Duarte, Fatima Zahra, Iris Wong, Omar Farouk.

Q3. Which projects are staffed by someone skilled in Python?
   Atlas, Guardrail.

Q4. Which departments collaborate with Data through shared projects?
   Engineering.

Q5. Who oversees the largest cross-functional team?
   Nadia Osei (CEO), whose organization spans 4 departments across 14 reports.


Every answer is a traversal result rendered as prose. The first answer follows the reporting edge up to a manager and back down to the manager's other reports. The second follows the reporting relationship transitively to its closure beneath an executive. The third passes from a skill through its holders to the projects they staff. The fourth connects two departments through a shared project. The fifth counts the departments beneath each manager and returns the maximum, which is the chief executive. None of these facts is stated in any single document, and each is produced by the same system whether it runs on the in-memory graph or on Neo4j, which is the outcome the chapter set out to demonstrate: a retrieval system whose native operation is the traversal that enterprise questions require.

## Extending the lab

The system as built answers structural questions. The natural extension, and the one that closes the loop with the earlier parts, is to add the semantic entry that the running question requires. A topic question first uses the document vector index to identify the projects a topic concerns, then traverses staffing and reporting exactly as the flagship query in Part 7 does. The graph-aware answering demonstrated in Part 5 is precisely this composition, and folding it into the system as a sixth method, one that accepts a free-text topic rather than a named entity, turns the capstone into the complete agentic and graph-based retriever the chapter has been building toward.

## Exercises

- Add a topic-query method that accepts free text, uses the vector index to find the relevant projects, and traverses to the engineering managers, reproducing the running question through the assembled system.
- Introduce a new employee and a new assignment, re-run the idempotent ingestion, and confirm that the five answers update without any other change to the system.
- Replace the deterministic verbalization with a call to the language model that receives the retrieved facts as context, and compare the phrasing while confirming the facts are unchanged.

## Summary

The lab assembled ingestion, embeddings, traversal, and verbalization into one GraphRAG system with a uniform interface, and answered five representative enterprise questions with grounded results that name the paths supporting them. The system runs unchanged on the in-memory graph and on Neo4j, differing only in the configured backend, and its answers are traversals that no flat retriever produces. Extending it with a free-text topic method that begins at the vector index and continues through the graph reproduces the running question and completes the architecture the chapter set out to build.

# Closing

This chapter traced a single enterprise question from the point where classical retrieval fails to a system that answers it correctly. The question, which engineering managers have team members working on projects involving LLM security, was chosen because a correct answer requires operations that a flat vector store cannot perform: a semantic judgment, a join, a traversal, and a filter, composed into one path. Each part supplied one capability. Part 1 established the paradigm limit. Part 2 built the agent that plans and routes. Parts 3 and 4 built the relational and semantic retrievers the agent routes to. Parts 5 through 7 built the graph, its backend, and the multi-hop traversal that the question ultimately reduces to. Part 8 placed these in a decision framework, and the lab assembled them into one system. The through-line is that retrieval architecture should follow the operations a workload performs, and that enterprise workloads perform joins, aggregations, and traversals that only structured and graph-based retrieval can serve.

## Key takeaways

1. Classical retrieval answers questions whose answers already exist as text and lie close to the query in embedding space, and it fails silently on questions whose answers are joins, aggregates, or paths that appear in no single passage.
2. Agentic retrieval replaces the fixed retrieve-then-generate pipeline with a loop that decomposes a question, routes each part to a suitable retriever, accumulates evidence, and reflects on sufficiency before answering.
3. Dynamic SQL generation lets a language model translate a natural-language subquestion into a query over structured tables, and it is safe for production only behind validation that restricts it to read-only, single-statement queries against permitted tables.
4. Dynamic vector retrieval answers the semantic part of a question, and production semantic search is a pipeline of optional filtering, dense and lexical scoring, rank-based fusion, and shortlist reranking rather than a single similarity lookup.
5. GraphRAG represents entities and relationships explicitly, so its unit of retrieval is a connected subgraph that preserves the paths compound questions traverse, which is the property flat retrieval lacks.
6. A property-graph database such as Neo4j aligns the storage engine with traversal, expresses hierarchy queries as pattern matches rather than recursion, and can serve semantic entry and structural traversal from one store through a native vector index.
7. Multi-hop reasoning composes several relationship hops into one path, and it is best expressed as a single compiled traversal when every hop is structural and as recursive retrieval only when a hop needs a capability outside the graph engine.
8. The right retrieval architecture follows from the operations the workload performs, and a capable enterprise system is usually a hybrid that routes exact subquestions to relational retrieval, semantic subquestions to a vector index, and relationship subquestions to a graph.

## Architecture recap

The complete system is an agent wrapped around three retrievers and a synthesizer. A question enters an intent-and-routing stage that decomposes it into typed subquestions. A planner orders the subquestions so that dependencies are respected, and for each it selects a retriever: the relational store for exact and aggregate subquestions, the vector index for semantic subquestions, and the graph for traversal subquestions. The retrievers write to a shared evidence buffer. A reflection stage judges whether the accumulated evidence answers the original question and, if not, returns control to the planner for another iteration. When the evidence is sufficient, a synthesis stage composes a grounded answer that cites the relationships and records supporting it. The graph is the component that makes the traversal subquestions answerable, the vector index makes the semantic subquestions answerable, and the relational store makes the exact subquestions answerable; the agent is the control structure that composes them, and the reflection loop is what lets the system recover from an insufficient first plan.

## Production-readiness checklist

- Classify the distribution of incoming questions and confirm that each retriever serves a meaningful share of it before that retriever is operated in production, so that architecture is justified by evidence rather than assumption.
- Guard every dynamically generated SQL query with validation that permits only read-only, single-statement queries against an allowlist of tables, executed on a least-privilege connection, so that a generated query can never mutate data or reach forbidden tables.
- Treat the embedding model version as part of the vector index identity, and re-embed the corpus when the model changes rather than mixing vectors produced by different models, so that similarity remains meaningful.
- Prefer a store that filters natively over post-filtering when filters are selective, so that metadata or label constraints narrow candidates before ranking rather than starving the result set.
- Build the graph through an idempotent, versioned, auditable ingestion pipeline driven by identifiable records, so that the loader can run repeatedly without duplicating data and stale nodes can be reconciled.
- Bound every transitive traversal to a depth appropriate to the domain and restrict it to the relationship types the question implicates, so that a multi-hop query cannot explore an unbounded number of paths.
- Evaluate the end-to-end system on a held-out set of questions that deliberately require joins and traversals, so that the silent failures of a mismatched retriever surface as wrong answers on those questions before users encounter them.
- Log retrieved evidence alongside the final answer, so that a wrong answer can be traced to the retrieval step that produced or omitted the deciding fact.
- Separate offline extraction and ingestion from online serving, so that query latency is predictable and the graph is an auditable artifact rather than mutable state edited during a query.

## Interview question bank

**1. Why does a retrieval system with a strong embedding model still fail on many enterprise questions?** Embedding quality governs only how well passages are ranked by topical similarity. Questions whose answers are joins, aggregates, exclusions, or multi-hop paths have no representation in any single passage, so no ranking of passages can surface them. The limit is the retrieval paradigm, not the model, and the remedy is to add structured or graph-based retrieval rather than to tune the embeddings.

**2. What distinguishes an agentic retriever from a fixed RAG pipeline?** A fixed pipeline retrieves once and generates. An agentic retriever plans retrieval as a sequence of steps, chooses a tool per step, executes them, inspects the accumulated evidence, and decides whether to continue, which turns retrieval into a control problem. This lets it answer compound questions whose parts require different retrieval mechanisms and different orderings.

**3. How would you make model-generated SQL safe to run against a production database?** Restrict it structurally and by privilege. Validate that the query is a single read-only statement, reject data-definition and data-manipulation keywords, confine it to an allowlist of tables and views, and execute it on a connection that has only read access to those objects. Add a row limit and a timeout so that a valid but expensive query cannot exhaust resources.

**4. Why is reciprocal rank fusion a robust way to combine lexical and dense retrieval?** It uses only the rank position of each document in each list, not the raw scores, so it is invariant to the differing and query-dependent score scales of a lexical and a dense retriever. That invariance removes the brittle normalization a weighted score sum would require and makes the fusion a dependable default with little tuning.

**5. What is the unit of retrieval in GraphRAG, and why does it matter?** The unit is a connected subgraph rather than an independent passage. Compound questions require facts that lie on a path through several entities, and a subgraph preserves that path as edges, whereas a ranked list of passages discards the connections between them. The change of unit is what makes traversal and multi-hop questions answerable.

**6. Where does a GraphRAG pipeline most often fail, and how do you detect it?** At extraction, where a missed or mistyped relationship removes a path the system needs, and the failure is silent because retrieval still returns something. Detection requires evaluating extraction against a labeled sample and evaluating end-to-end answers on questions that deliberately require multi-hop traversal, so that missing edges appear as wrong answers rather than going unnoticed.

**7. Why store an organizational hierarchy in a graph database rather than a relational one?** Because the dominant operation is traversal. A variable-length relationship pattern in Cypher states the shape of the path and the engine matches it directly, whereas a relational store expresses the same traversal as a recursive common table expression built from repeated self-joins. The graph formulation aligns the query with the storage model and is clearer and, for deep traversals, more efficient.

**8. What does a native vector index in a graph database provide that separate stores do not?** It lets one system serve semantic entry and structural traversal together, so a query can locate entry nodes by embedding similarity and then expand along relationships without moving data between systems. Because label and property constraints narrow candidates natively, it also avoids the post-filtering starvation that afflicts a vector store queried with a selective external filter.

**9. When should a multi-hop question be a single traversal, and when should it be recursive retrieval?** When every hop is a structural relationship the graph stores, a single compiled traversal is preferable because the database executes it in one efficient, atomic pass. When a hop requires a capability outside the graph engine, most often a semantic judgment that must consult a model or a vector index, that hop must be a separate retrieval whose result seeds the next, which is recursive retrieval and the graph-aware form of the agent loop.

**10. What is the danger of an unbounded variable-length traversal in production?** Its cost grows with the number of paths it can explore, and on a densely connected graph that number can be enormous, so an unbounded traversal risks exhausting time or memory and destabilizing the database. Bounding the depth to the maximum plausible for the domain and restricting the relationship types followed keeps the search finite and its cost predictable.

**11. How do you choose among relational, vector, and graph retrieval for a question?** By the operation a correct answer requires. An answer that is an exact function of typed columns is relational, an answer that depends on the meaning of unstructured text is vector, and an answer that is a traversal or a path is graph. A question whose parts fall into different classes is hybrid and is served by routing each part to the appropriate retriever and composing the results rather than forcing the whole question through one.

**12. When is the added cost of a hybrid retrieval architecture justified?** When the workload contains questions that genuinely combine types, such as finding documents about a topic that also satisfy a structural constraint, and when those questions occur in enough volume to warrant the operational overhead of running and fusing several subsystems. The justification is evidence from the observed question distribution, not a general presumption that more machinery is better.

## Additional resources

Several bodies of work extend the material of this chapter. Microsoft Research's publications and open-source project on GraphRAG develop the construction of a knowledge graph from a corpus and its use for global summarization and multi-hop question answering. Neo4j's documentation on Cypher, on indexing and the native vector index, and on the Graph Data Science library covers the query language and the structural algorithms used here at a depth beyond this chapter. The LlamaIndex and LangGraph documentation describe production patterns for agentic and graph-based retrieval, including the planning and reflection loops of Part 2. The Sentence-Transformers documentation covers the embedding models and cross-encoders used for the semantic pipeline of Part 4, and the literature on BM25 and on reciprocal rank fusion provides the foundations of the lexical and fusion components. For the enterprise context, public descriptions of large professional-network and code-search systems illustrate at scale the graph and semantic retrieval patterns this chapter demonstrates in miniature.

## Advanced and stretch exercises

- Replace the deterministic offline fallbacks with live model calls, instrument how often the planner and the SQL generator require self-correction, and analyze the questions that trigger a second attempt.
- Add a reflection loop that re-plans when a retriever returns empty evidence, and validate it on a question whose first plan is deliberately insufficient so that recovery is exercised.
- Extend the graph with temporal validity on relationships, and answer a question about the state of the organization at a past date, which requires filtering traversals by an as-of time.
- Benchmark a bounded multi-hop traversal on synthetic graphs of increasing size and connectivity, and characterize query latency as a function of depth and branching factor.
- Introduce a second embedding model, build an evaluation that detects recall drift between the two on a fixed query set, and use it to decide whether a model upgrade is safe.
- Fold a free-text topic method into the assembled system that begins at the vector index and continues through the graph, reproducing the running question end to end through one interface.